# VID_FSHOT - Few-shot multimodal para EXIST 2026 (VIDEOS)

Pipeline **few-shot puro** (sin fine-tuning) sobre los TikToks del reto. Genera salidas **HARD** PyEvALL para las tres subtasks **T3.1, T3.2 y T3.3** y empaqueta la submission.

Soporta dos VLMs intercambiables (selector `BACKEND` en la celda 1.2):

| BACKEND | Modelo HuggingFace | Notas |
|---------|--------------------|-------|
| `"qwen"`  | `Qwen/Qwen3.5-27B` | VLM unificado nativo (~27B, Feb 2026); usa `qwen_vl_utils` y los parametros `max_pixels` / `fps` |
| `"gemma"` | `google/gemma-4-31B-it`     | multimodal nativo (texto+imagen+video+audio); el processor se encarga de leer el video |

Ambos backends comparten:
- Mismos prompts del lab guidelines (T3.1, T3.2, T3.3)
- Mismos few-shot pools en disco (`cache_<backend>_vid_fshot/few_shot_pools/`)
- Mismo loop jerarquico T3.1 -> T3.2/T3.3 con checkpointing
- Mismo formato de submission PyEvALL

Las cachés de predicciones, checkpoints y métricas se guardan **separadas por backend** para poder comparar runs sin pisar resultados.

## Cambios respecto a la version inicial
- Pool con estratificacion bilingue ES/EN y diversificacion 70/30 high/medium consensus en T3.1/T3.2.
- Pool T3.3 con 2 ejemplos extra de co-ocurrencia multi-label (total 12).
- `USE_FEW_SHOT_RATIONALE`: shots con rationale corta (UNA frase, opcional).
- Smoke test que cuenta tokens reales del prompt y avisa si > 80% / aborta si > 100% del max_ctx.
- Sample sanity ESTRATIFICADO por t31_hard (sklearn).
- Baseline majority comparado en `summary_df` (`run="majority"`).
- Variabilidad multi-seed del pool (`N_POOL_SEEDS > 1` -> mean +/- std).
- Reporte de fallbacks por subtask (parser-miss vs exception).
- VRAM cleanup periodico (`empty_cache + gc.collect`) y excepciones tipadas (OOM, FileNotFound, RuntimeError, Exception).
- `T33_FALLBACK_CATEGORY` configurable (default = moda real del TRAIN).


## 1 - Setup

In [1]:
# === 1.1 - Deteccion Colab vs local ==========================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("IN_COLAB =", IN_COLAB)


Mounted at /content/drive
IN_COLAB = True


In [2]:
!pip install -U "git+https://github.com/huggingface/transformers.git"

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-c0k5mk0b
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-c0k5mk0b
  Resolved https://github.com/huggingface/transformers.git to commit ddb841f48888e3fcf50c3f2a570ac9774aa7373c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11874752 sha256=3dde572ee742b4267cee9ee34b7779514bbce30bb9e6e1bc87ffa057cd4f2d61
  Stored in directory: /tmp/pip-ephem-wheel-cache-uqvl7_70/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
# === 1.2 - Seleccion de backend (modelo VLM) ================================
# Cambia BACKEND para alternar entre los dos VLMs (ambos instruct, sin thinking):
#   "qwen"  -> Qwen/Qwen3.5-27B            (VLM unificado nativo, ~27B, Feb 2026)
#   "gemma" -> google/gemma-4-31B-it       (multimodal denso 31B)
BACKEND = "gemma"   # "qwen" | "gemma"

# Cada backend define modelo, clase de transformers, paquetes que necesita y
# max_new_tokens por subtask. Ambos son instruct puros: responden la etiqueta
# directamente sin bloque <think>...</think>, asi que los presupuestos de
# tokens son cortos.
BACKEND_CONFIG = {
    "qwen": {
        # Qwen3.5-27B es VLM unificado (texto+imagen+video). La clase concreta
        # puede no estar expuesta segun la version de transformers; usamos
        # AutoModelForImageTextToText, que esta registrada en >=4.57 y
        # despacha al modelo correcto leyendo config.json.
        "model_id"        : "Qwen/Qwen3.5-27B",
        "model_class"     : "AutoModelForImageTextToText",
        "needs_qwen_utils": True,
        "max_new_tokens"  : {"t31": 512, "t32": 512, "t33": 1024},
    },
    "gemma": {
        "model_id"        : "google/gemma-4-31B-it",
        "model_class"     : "AutoModelForMultimodalLM",
        "needs_qwen_utils": False,
        "max_new_tokens"  : {"t31": 512, "t32": 512, "t33": 1024},
    },
}
assert BACKEND in BACKEND_CONFIG, f"BACKEND desconocido: {BACKEND}"
BCFG = BACKEND_CONFIG[BACKEND]

# === Few-shot extras (compartidos entre backends) ===========================
# Si True, los shots del few-shot pool incluyen la etiqueta seguida de un
# guion y una rationale corta sacada de las reglas del system prompt
# (T3.1: 1-12, T3.2: D1-D8 / J1-J6, T3.3: las 5 categorias).
# Si activas este flag con Gemma, plantea subir max_new_tokens — el modelo
# tendera a imitar el formato y necesitara mas espacio para responder.
USE_FEW_SHOT_RATIONALE = False

# Variabilidad multi-seed del pool. Por defecto 1 (no rompe el flujo).
# Con N_POOL_SEEDS > 1 la celda final 8.5 reconstruye el pool con seeds
# distintas, corre la inferencia y reporta media +- std de ICM/ICMNorm/FMeasure.
N_POOL_SEEDS = 1

# Categoria de fallback para T3.3 cuando el parser no extrae nada / hay error.
# None -> se calcula automaticamente como la moda real del split TRAIN
# (en cell 2.1 una vez se carga el dataset).
T33_FALLBACK_CATEGORY = None

print(f"BACKEND  = {BACKEND}")
print(f"MODEL_ID = {BCFG['model_id']}")
print(f"max_new_tokens = {BCFG['max_new_tokens']}")
print(f"USE_FEW_SHOT_RATIONALE = {USE_FEW_SHOT_RATIONALE}  | N_POOL_SEEDS = {N_POOL_SEEDS}")
print(f"T33_FALLBACK_CATEGORY  = {T33_FALLBACK_CATEGORY}  (None -> auto-mode TRAIN)")


BACKEND  = gemma
MODEL_ID = google/gemma-4-31B-it
max_new_tokens = {'t31': 512, 't32': 512, 't33': 1024}
USE_FEW_SHOT_RATIONALE = False  | N_POOL_SEEDS = 1
T33_FALLBACK_CATEGORY  = None  (None -> auto-mode TRAIN)


In [3]:
# === 1.3 - Dependencias (conditional sobre BACKEND) =========================
import sys, subprocess, importlib

# Comunes a ambos backends
COMMON_PKGS = [
    "transformers>=4.57",   # Qwen3.5 y Gemma-4 requieren >=4.57
    "accelerate",
    "pillow",
    "tqdm",
    "scikit-learn",
]
QWEN_PKGS  = ["qwen-vl-utils[decord]"]   # decord para leer mp4 frame-a-frame
GEMMA_PKGS = ["torchvision", "torchcodec"]  # torchcodec lee video para Gemma

PKGS = COMMON_PKGS + (QWEN_PKGS if BACKEND == "qwen" else GEMMA_PKGS)
for p in PKGS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

# bitsandbytes solo si vamos a quantizar (GPUs con < 24GB VRAM)
try:
    import bitsandbytes  # noqa
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
    except Exception as e:
        print("[warn] bitsandbytes no se pudo instalar:", e)

# Verificacion de imports criticos especificos del backend
if BACKEND == "qwen":
    try:
        importlib.import_module("qwen_vl_utils"); importlib.import_module("decord")
        print("OK: qwen_vl_utils + decord disponibles")
    except ImportError as e:
        print(f"[warn] {e}, reinstalando con [decord]...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-cache-dir", "qwen-vl-utils[decord]"])
        importlib.invalidate_caches()
        import qwen_vl_utils, decord  # noqa
        print("OK tras reinstalar")
else:  # gemma
    try:
        importlib.import_module("torchcodec")
        print("OK: torchcodec disponible para video en Gemma-4")
    except ImportError as e:
        print(f"[warn] {e}, reinstalando torchcodec...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-cache-dir", "torchcodec"])
        importlib.invalidate_caches()
        import torchcodec  # noqa
        print("OK tras reinstalar")


KeyboardInterrupt: 

In [4]:
# === 1.4 - Imports + reproducibilidad ========================================
import json, os, random, gc, warnings, re, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")


DEVICE: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM: 102.0 GB


In [5]:
# === 1.5 - Rutas + workflow (cache separada por BACKEND) ====================
# Workflow de dos etapas:
#   STAGE = "sanity"     -> autoeval rapida sobre SANITY_N videos de TRAIN.
#   STAGE = "submission" -> run completo sobre los 674 videos de TEST oficial.
STAGE = "submission"          # "sanity"  |  "submission"
SANITY_N = 200            # Videos de TRAIN para sanity (~10-30s/video)

if STAGE == "sanity":
    RUN_MODE       = "train_eval"
    MAX_EVAL_VIDS  = SANITY_N
elif STAGE == "submission":
    RUN_MODE       = "test"
    MAX_EVAL_VIDS  = None
else:
    raise ValueError(f"STAGE desconocido: {STAGE}")

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/EXIST_2026")
else:
    PROJECT_ROOT = Path(r"c:/Users/juana/OneDrive/Escritorio/TFG/EXIST_2026")

DATASET_DIR = PROJECT_ROOT / "EXIST 2026 Dataset V0.1"

TRAIN_JSON_CANDIDATES = [
    DATASET_DIR / "EXIST 2026 Videos Dataset" / "training" / "EXIST2026_training.json",
    DATASET_DIR / "EXIST2026_training_videos.json",
]
TRAIN_VID_DIR_CANDIDATES = [
    DATASET_DIR / "EXIST 2026 Videos Dataset" / "training" / "videos",
    DATASET_DIR / "videos",
]
TEST_JSON_CANDIDATES = [
    DATASET_DIR / "Dataset - Evall" / "EXIST 2026 Videos Dataset" / "test" / "EXIST2026_test_clean.json",
    PROJECT_ROOT / "Dataset - Evall" / "EXIST 2026 Videos Dataset" / "test" / "EXIST2026_test_clean.json",
]
TEST_VID_DIR_CANDIDATES = [
    DATASET_DIR / "Dataset - Evall" / "EXIST 2026 Videos Dataset" / "test" / "videos",
    PROJECT_ROOT / "Dataset - Evall" / "EXIST 2026 Videos Dataset" / "test" / "videos",
]

def _first_existing(cands):
    for c in cands:
        if c.exists(): return c
    return None

TRAIN_JSON     = _first_existing(TRAIN_JSON_CANDIDATES)
TRAIN_VID_DIR  = _first_existing(TRAIN_VID_DIR_CANDIDATES)
TEST_JSON      = _first_existing(TEST_JSON_CANDIDATES)
TEST_VID_DIR   = _first_existing(TEST_VID_DIR_CANDIDATES)

assert TRAIN_JSON is not None,    f"No se encontro JSON train: {TRAIN_JSON_CANDIDATES}"
assert TRAIN_VID_DIR is not None, f"No se encontro dir train videos: {TRAIN_VID_DIR_CANDIDATES}"
if RUN_MODE == "test":
    assert TEST_JSON is not None,    f"No se encontro JSON test: {TEST_JSON_CANDIDATES}"
    assert TEST_VID_DIR is not None, f"No se encontro dir test videos: {TEST_VID_DIR_CANDIDATES}"

# Cache separada por backend para no pisar resultados al cambiar de modelo
CACHE_DIR = PROJECT_ROOT / f"cache_{BACKEND}_vid_fshot"
POOLS_DIR = CACHE_DIR / "few_shot_pools"
PRED_DIR  = CACHE_DIR / "predictions"
CKPT_DIR  = CACHE_DIR / "checkpoints"
METR_DIR  = CACHE_DIR / "metrics"
LOGS_DIR  = CACHE_DIR / "logs"
PYE_WORK  = CACHE_DIR / "pyevall_work"
SUBM_DIR  = CACHE_DIR / "submission"
for d in [CACHE_DIR, POOLS_DIR, PRED_DIR, CKPT_DIR, METR_DIR, LOGS_DIR, PYE_WORK, SUBM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"BACKEND        : {BACKEND}")
print(f"STAGE          : {STAGE}")
print(f"RUN_MODE       : {RUN_MODE}")
print(f"MAX_EVAL_VIDS  : {MAX_EVAL_VIDS}")
print(f"TRAIN_JSON     : {TRAIN_JSON}")
print(f"TRAIN_VID_DIR  : {TRAIN_VID_DIR} | count={len(list(TRAIN_VID_DIR.glob('*.mp4')))}")
if RUN_MODE == "test":
    print(f"TEST_JSON      : {TEST_JSON}")
    print(f"TEST_VID_DIR   : {TEST_VID_DIR} | count={len(list(TEST_VID_DIR.glob('*.mp4')))}")
print(f"CACHE_DIR      : {CACHE_DIR}")


BACKEND        : gemma
STAGE          : submission
RUN_MODE       : test
MAX_EVAL_VIDS  : None
TRAIN_JSON     : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/EXIST2026_training.json
TRAIN_VID_DIR  : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/videos | count=2524
TEST_JSON      : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/Dataset - Evall/EXIST 2026 Videos Dataset/test/EXIST2026_test_clean.json
TEST_VID_DIR   : /content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/Dataset - Evall/EXIST 2026 Videos Dataset/test/videos | count=674
CACHE_DIR      : /content/drive/MyDrive/EXIST_2026/cache_gemma_vid_fshot


In [6]:
# === 1.6 - Carga del modelo VLM (dispatch por BACKEND) ======================
import transformers
from transformers import AutoProcessor

print("transformers version:", transformers.__version__)

# Quantizamos a 4-bit cuando la GPU es modesta (< 24 GB VRAM).
USE_4BIT = (torch.cuda.is_available() and
            torch.cuda.get_device_properties(0).total_memory < 24 * 1e9)
print("USE_4BIT =", USE_4BIT)

def _resolve_model_class(class_name):
    """Localiza la clase de modelo dentro de transformers."""
    cls = getattr(transformers, class_name, None)
    if cls is None:
        raise ImportError(
            f"{class_name} no esta disponible en transformers {transformers.__version__}. "
            f"Actualiza transformers (>=4.57)."
        )
    return cls

def _load_one(model_id, class_name):
    """Intenta cargar un (model_id, class_name) concreto. Devuelve (model, processor)
    o lanza la excepcion."""
    cls = _resolve_model_class(class_name)
    kwargs = dict(device_map="auto")
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )
    else:
        kwargs["dtype"] = torch.bfloat16
    m = cls.from_pretrained(model_id, **kwargs)
    p = AutoProcessor.from_pretrained(model_id)
    return m, p

# Carga unica del modelo seleccionado (sin fallback: ambos backends son instruct
# y no necesitan variantes alternativas).
MODEL_ID = BCFG["model_id"]
MODEL_CLASS_NAME = BCFG["model_class"]
model, processor = _load_one(MODEL_ID, MODEL_CLASS_NAME)
print(f"OK Modelo cargado: {MODEL_ID}  (4bit={USE_4BIT})")
model.eval()
# Ambos backends son instruct puros (sin razonamiento <think>...</think>).
# IS_THINKING se mantiene como flag por compatibilidad pero siempre es False.
IS_THINKING = False
print(f"Modelo en eval mode | IS_THINKING={IS_THINKING}")


transformers version: 5.8.0.dev0
USE_4BIT = False


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

...

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

OK Modelo cargado: google/gemma-4-31B-it  (4bit=False)
Modelo en eval mode | IS_THINKING=False


## 2 - Carga de datos + hard labels

Umbrales oficiales (lab guidelines V0.5 pag. 17):
"Due to the nature of subtasks 3.1, 3.2 and 3.3 and the complexity of video labeling [...] hard labels included are those annotated by **more than 1 annotator**".

Por tanto **las tres subtasks de video usan threshold > 1**, no como memes (>3, >2, >1).
Tambien T3.1 admite la label `UNKNOWN` que se descarta antes de calcular consenso.

In [7]:
# === 2.1 - Hard label helpers + carga JSON (train + test unificados) ========
SEXISM_CATS = [
    "IDEOLOGICAL-INEQUALITY", "STEREOTYPING-DOMINANCE", "OBJECTIFICATION",
    "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE",
]
T31_INT_TO_STR = {0: "NO", 1: "YES"}
T32_INT_TO_STR = {0: "NO", 1: "DIRECT", 2: "JUDGEMENTAL"}
T32_STR_TO_INT = {v: k for k, v in T32_INT_TO_STR.items()}

# Threshold oficial > 1 voto para las 3 subtasks de video
def hard_t31(labs):
    """T3.1 -> int 0/1 o None. UNKNOWN se descarta. Threshold > 1 voto."""
    valid = [v for v in labs if v != "UNKNOWN"]
    if not valid: return None
    yes = sum(1 for v in valid if v == "YES")
    no  = sum(1 for v in valid if v == "NO")
    if yes > 1 and yes >= no: return 1
    if no  > 1 and no  >= yes: return 0
    return None

def hard_t32(labs):
    """T3.2 -> int 0/1/2 o None. Mapea '-' -> 'NO'; threshold > 1."""
    clean = [l for l in labs if l != "UNKNOWN"]
    if not clean: return None
    mapped = ["NO" if l == "-" else l for l in clean]
    c = Counter(mapped)
    top, votes = c.most_common(1)[0]
    if votes <= 1: return None
    return T32_STR_TO_INT[top]

def hard_t33_multihot(arrs):
    """T3.3 hard -> multi-hot (5,) con 0/1. Cat activa si > 1 anotador."""
    c = Counter()
    for arr in arrs:
        if "UNKNOWN" in arr: continue
        for l in arr:
            if l != "UNKNOWN": c[l] += 1
    vec = np.zeros(len(SEXISM_CATS), dtype=np.float32)
    for i, cat in enumerate(SEXISM_CATS):
        if c.get(cat, 0) > 1: vec[i] = 1.0
    return vec

def _build_rows(raw_dict, vid_dir, split):
    out = []
    for vid, m in raw_dict.items():
        vid_name = m.get("video") or f"{vid}.mp4"
        vid_path = vid_dir / vid_name
        labs31 = m.get("labels_task3_1") or []
        labs32 = m.get("labels_task3_2") or []
        arrs33 = m.get("labels_task3_3") or []
        out.append({
            "id_EXIST" : str(vid),
            "split"    : split,
            "lang"     : m.get("lang", "en"),
            "text_ocr" : (m.get("text") or "").strip(),
            "video_path": str(vid_path),
            "vid_ok"   : vid_path.exists(),
            "t31_hard" : hard_t31(labs31) if labs31 else None,
            "t32_hard" : hard_t32(labs32) if labs32 else None,
            "t33_hard" : hard_t33_multihot(arrs33) if arrs33 else
                         np.zeros(len(SEXISM_CATS), dtype=np.float32),
        })
    return out

# ---- Carga TRAIN (siempre necesario para construir el few-shot pool) -------
with open(TRAIN_JSON, "r", encoding="utf-8", errors="replace") as f:
    raw_train = json.load(f)
rows_train = _build_rows(raw_train, TRAIN_VID_DIR, "train")

# ---- Carga TEST (solo si RUN_MODE=='test') ---------------------------------
raw_test, rows_test = {}, []
if RUN_MODE == "test":
    with open(TEST_JSON, "r", encoding="utf-8", errors="replace") as f:
        raw_test = json.load(f)
    rows_test = _build_rows(raw_test, TEST_VID_DIR, "test")

df  = pd.DataFrame(rows_train + rows_test)
raw = {**raw_train, **raw_test}

df_train = df[df["split"] == "train"].reset_index(drop=True)
df_test  = df[df["split"] == "test"].reset_index(drop=True)

print(f"Videos TRAIN cargados     : {len(df_train):,}")
print(f"  videos encontrados      : {df_train['vid_ok'].sum():,}")
print(f"  t31_hard valida (>1)    : {df_train['t31_hard'].notna().sum():,}")
print(f"  t32_hard valida (>1)    : {df_train['t32_hard'].notna().sum():,}")
if RUN_MODE == "test":
    print(f"\nVideos TEST cargados      : {len(df_test):,}")
    print(f"  videos encontrados      : {df_test['vid_ok'].sum():,}")

assert df_train["vid_ok"].all(), "Hay videos train sin .mp4; revisa TRAIN_VID_DIR"
if RUN_MODE == "test":
    assert df_test["vid_ok"].all(), "Hay videos test sin .mp4; revisa TEST_VID_DIR"

# === T33_FALLBACK_CATEGORY: si no esta seteado, lo derivamos como la categoria
# T3.3 mas frecuente en TRAIN (entre los videos sexistas con hard-label valida).
if T33_FALLBACK_CATEGORY is None:
    _t33_counts = Counter()
    for vec in df_train["t33_hard"]:
        if vec is None: continue
        for k, cat in enumerate(SEXISM_CATS):
            if vec[k] > 0.5:
                _t33_counts[cat] += 1
    T33_FALLBACK_CATEGORY = (_t33_counts.most_common(1)[0][0]
                              if _t33_counts else "STEREOTYPING-DOMINANCE")
print(f"T33_FALLBACK_CATEGORY (auto-mode TRAIN) = {T33_FALLBACK_CATEGORY}")


Videos TRAIN cargados     : 2,524
  videos encontrados      : 2,524
  t31_hard valida (>1)    : 2,508
  t32_hard valida (>1)    : 2,466

Videos TEST cargados      : 674
  videos encontrados      : 674
T33_FALLBACK_CATEGORY (auto-mode TRAIN) = STEREOTYPING-DOMINANCE


## 3 - Few-shot pool (anti-leakage, alta consenso)

Pools mas pequenos que en memes porque cada video del pool ocupa MUCHA VRAM (varios frames). Diseno hierarchical-conditional: T3.2/T3.3 reciben solo videos ya marcados como sexistas, asi que sus pools NO incluyen NO.

- T3.1: 2 ejemplos (1 YES + 1 NO).
- T3.2: 2 ejemplos (1 DIRECT + 1 JUDGEMENTAL).
- T3.3: 5 ejemplos (1 por cat sexista, single-label preferentemente).

In [8]:
# === 3.1 - Builder de few-shot pools de VIDEO ================================
# Cambios respecto al builder anterior:
#   (1) Estratificacion BILINGUE (mitad ES / mitad EN) dentro de cada clase.
#   (2) Diversificacion de consenso en T3.1 y T3.2:
#         70% ejemplos high-consensus + 30% medium-consensus (cons in [0.5, 0.7]).
#       En T3.3 se mantiene la logica single-label/cons (no se toca).
#   (3) En T3.3, tras seleccionar N por categoria, se anaden N_T33_COOCCUR
#       ejemplos extra con multi-label real (>=2 categorias activas, alto
#       consenso). Total final del pool T3.3: 5*N_PER_CLASS_T33 + N_T33_COOCCUR.
#   (4) Soporte multi-seed: la funcion principal acepta `seed`. Con seed=None
#       se ordena de forma determinista por id; con seed entero se usa un
#       tiebreaker aleatorio para empates de consenso, lo que produce pools
#       distintos por semilla (lo necesita la celda 8.5 multi-seed).

N_PER_CLASS_T31 = 3   # pool 6 (3 YES + 3 NO)
N_PER_CLASS_T32 = 3   # pool 6 (3 DIRECT + 3 JUDGEMENTAL)
N_PER_CLASS_T33 = 2   # 2 por cat sexista
N_T33_COOCCUR   = 2   # extra ejemplos multi-label
HIGH_CONS_FRAC  = 0.7
MED_CONS_RANGE  = (0.5, 0.7)

def _get_lang(raw_pool, vid):
    return (raw_pool[str(vid)].get("lang") or "en").lower()[:2]

def _consensus_t31(labs):
    valid = [v for v in labs if v != "UNKNOWN"]
    if not valid: return 0.0
    return Counter(valid).most_common(1)[0][1] / len(valid)

def _consensus_t32(labs):
    valid = [l for l in labs if l != "UNKNOWN"]
    if not valid: return 0.0
    mapped = ["NO" if l == "-" else l for l in valid]
    return Counter(mapped).most_common(1)[0][1] / len(mapped)

def _split_high_med(cands_sorted, n_total,
                     high_frac=HIGH_CONS_FRAC, med_range=MED_CONS_RANGE):
    """Toma cands ordenados por consensus desc y devuelve hasta n_total
    ejemplos con `high_frac` proporcion high-consensus (>= med_range[1]) y
    el resto medium-consensus (cons in med_range). Si falta de alguna banda,
    rellena con la otra hasta llegar a n_total."""
    if n_total <= 0: return []
    n_high = max(1, round(n_total * high_frac))
    n_med  = max(0, n_total - n_high)
    high   = [c for c in cands_sorted if c[2] >= med_range[1]]
    med    = [c for c in cands_sorted if med_range[0] <= c[2] < med_range[1]]
    out = high[:n_high] + med[:n_med]
    if len(out) < n_total:
        used = {c[0] for c in out}
        for c in cands_sorted:
            if c[0] not in used:
                out.append(c); used.add(c[0])
                if len(out) >= n_total: break
    return out[:n_total]

def _stratify_by_lang(cands_sorted, n_total):
    """Estratifica un set de candidatos en mitad ES / mitad EN. Si una de
    las dos lenguas no tiene suficiente, completa con la otra."""
    if n_total <= 0 or not cands_sorted: return []
    by = {"es": [], "en": []}
    for c in cands_sorted:
        lang = c[3] if len(c) >= 4 else "en"
        by.setdefault(lang, []).append(c)
    n_es = n_total // 2
    n_en = n_total - n_es
    out = (by.get("es", [])[:n_es]) + (by.get("en", [])[:n_en])
    if len(out) < n_total:
        used = {c[0] for c in out}
        for c in cands_sorted:
            if c[0] not in used:
                out.append(c); used.add(c[0])
                if len(out) >= n_total: break
    return out[:n_total]

def _sort_with_seed(cands, seed):
    """Ordena por consensus desc; con seed entero anade tiebreaker aleatorio."""
    if seed is None:
        return sorted(cands, key=lambda x: (-x[2], x[0]))
    rng = random.Random(int(seed))
    aug = [(*c, rng.random()) for c in cands]
    aug.sort(key=lambda x: (-x[2], x[-1]))
    return [tuple(t[:-1]) for t in aug]

def select_few_shot_pool_t31(df_pool, raw_pool, n_per=N_PER_CLASS_T31, seed=None):
    """Pool T3.1: para cada clase (YES, NO) selecciona n_per con balance
    bilingue ES/EN y mezcla 70% high-cons + 30% medium-cons."""
    cands = []  # (id, label_int, cons, lang)
    for _, row in df_pool.iterrows():
        if pd.isna(row["t31_hard"]) or not row["vid_ok"]: continue
        cons = _consensus_t31(raw_pool[row["id_EXIST"]].get("labels_task3_1") or [])
        lang = _get_lang(raw_pool, row["id_EXIST"])
        cands.append((row["id_EXIST"], int(row["t31_hard"]), cons, lang))
    cands_sorted = _sort_with_seed(cands, seed)
    pool = []
    for cls in (1, 0):
        cls_cands = [c for c in cands_sorted if c[1] == cls]
        # 1) diversificacion de consenso
        diversified = _split_high_med(cls_cands, n_per)
        # 2) re-balanceo bilingue dentro de la mitad seleccionada por consenso
        # Reordenamos sobre el conjunto cls_cands aplicando bilingue, pero
        # respetando que vengan de los buckets high/med ya escogidos:
        diverse_ids = {c[0] for c in diversified}
        cls_cands_in_diverse = [c for c in cls_cands if c[0] in diverse_ids]
        balanced = _stratify_by_lang(cls_cands_in_diverse, n_per)
        if len(balanced) < n_per:
            # backfill bilingue con cls_cands enteros
            balanced = _stratify_by_lang(cls_cands, n_per)
        for vid, lbl, cons, lang in balanced:
            pool.append({"id": vid, "label": T31_INT_TO_STR[lbl],
                          "consensus": float(cons), "lang": lang})
    return pool

def select_few_shot_pool_t32(df_pool, raw_pool, n_per=N_PER_CLASS_T32, seed=None):
    """Pool T3.2: SOLO DIRECT y JUDGEMENTAL (hierarchical-conditional).
    Mismo esquema bilingue + consenso diversificado que T3.1."""
    cands = []
    for _, row in df_pool.iterrows():
        if pd.isna(row["t32_hard"]) or not row["vid_ok"]: continue
        cls = int(row["t32_hard"])
        if cls == 0: continue
        cons = _consensus_t32(raw_pool[row["id_EXIST"]].get("labels_task3_2") or [])
        lang = _get_lang(raw_pool, row["id_EXIST"])
        cands.append((row["id_EXIST"], cls, cons, lang))
    cands_sorted = _sort_with_seed(cands, seed)
    pool = []
    for cls in (1, 2):  # DIRECT, JUDGEMENTAL
        cls_cands = [c for c in cands_sorted if c[1] == cls]
        diversified = _split_high_med(cls_cands, n_per)
        diverse_ids = {c[0] for c in diversified}
        cls_cands_in_diverse = [c for c in cls_cands if c[0] in diverse_ids]
        balanced = _stratify_by_lang(cls_cands_in_diverse, n_per)
        if len(balanced) < n_per:
            balanced = _stratify_by_lang(cls_cands, n_per)
        for vid, lbl, cons, lang in balanced:
            pool.append({"id": vid, "label": T32_INT_TO_STR[lbl],
                          "consensus": float(cons), "lang": lang})
    return pool

def _select_t33_cooccurrence(df_pool, raw_pool, used_ids, n=N_T33_COOCCUR, seed=None):
    """Selecciona ejemplos T3.3 con >= 2 categorias activas y alto consenso."""
    cands = []
    for _, row in df_pool.iterrows():
        vid = row["id_EXIST"]
        if vid in used_ids or not row["vid_ok"]: continue
        multi = row["t33_hard"]
        if multi is None: continue
        active = [SEXISM_CATS[k] for k in range(len(SEXISM_CATS)) if multi[k] > 0.5]
        if len(active) < 2: continue
        arrs = raw_pool[vid].get("labels_task3_3") or []
        valid = [a for a in arrs if "UNKNOWN" not in a]
        if not valid: continue
        cons_per_cat = []
        for cat in active:
            n_with = sum(1 for a in valid if cat in a)
            cons_per_cat.append(n_with / len(valid))
        mean_cons = sum(cons_per_cat) / len(cons_per_cat)
        if mean_cons < 0.5: continue
        lang = _get_lang(raw_pool, vid)
        cands.append((vid, active, mean_cons, lang, len(active)))
    if seed is None:
        cands.sort(key=lambda x: (-x[4], -x[2], x[0]))
    else:
        rng = random.Random(int(seed) + 7)
        aug = [(*c, rng.random()) for c in cands]
        aug.sort(key=lambda x: (-x[4], -x[2], x[-1]))
        cands = [t[:-1] for t in aug]
    return [{"id": vid, "labels": cats, "consensus": float(cons), "lang": lang}
            for (vid, cats, cons, lang, _) in cands[:n]]

def select_few_shot_pool_t33(df_pool, raw_pool, n_per=N_PER_CLASS_T33, seed=None):
    """Pool T3.3: 2 ejemplos por categoria sexista (single-label preferred,
    bilingue ES/EN) + 2 ejemplos extra de co-ocurrencia multi-label."""
    pool, used = [], set()
    for ci, cat in enumerate(SEXISM_CATS):
        cands = []
        for _, row in df_pool.iterrows():
            if row["id_EXIST"] in used or not row["vid_ok"]: continue
            multi = row["t33_hard"]
            if multi is None or multi[ci] < 0.5: continue
            arrs = raw_pool[row["id_EXIST"]].get("labels_task3_3") or []
            valid = [a for a in arrs if "UNKNOWN" not in a]
            if not valid: continue
            n_single = sum(1 for a in valid if cat in a and len(a) == 1)
            n_with   = sum(1 for a in valid if cat in a)
            single   = n_single / len(valid)
            cons     = n_with   / len(valid)
            score    = single * 0.6 + cons * 0.4
            lang     = _get_lang(raw_pool, row["id_EXIST"])
            cands.append((row["id_EXIST"], score, cons, lang))
        if seed is None:
            cands.sort(key=lambda x: (-x[1], x[0]))
        else:
            rng = random.Random(int(seed) + ci)
            aug = [(*c, rng.random()) for c in cands]
            aug.sort(key=lambda x: (-x[1], x[-1]))
            cands = [t[:-1] for t in aug]
        # bilingue dentro de los top-K
        topk = cands[:max(n_per * 4, n_per)]
        # Adaptamos al formato (id, _, _, lang) que _stratify_by_lang espera
        as_lang = [(c[0], None, c[1], c[3]) for c in topk]
        balanced = _stratify_by_lang(as_lang, n_per)
        for vid, _none, score, lang in balanced:
            row_match = df_pool.loc[df_pool["id_EXIST"] == vid].iloc[0]
            multi = row_match["t33_hard"]
            picks = [SEXISM_CATS[k] for k in range(5) if multi[k] > 0.5]
            cons  = next((c[2] for c in cands if c[0] == vid), 0.0)
            pool.append({"id": vid, "labels": picks, "consensus": float(cons),
                          "lang": lang})
            used.add(vid)
    # Co-ocurrencia (multi-label real)
    pool.extend(_select_t33_cooccurrence(df_pool, raw_pool, used, n=N_T33_COOCCUR, seed=seed))
    return pool

POOL_PATHS = {
    "t31": POOLS_DIR / "few_shot_pool_t31.json",
    "t32": POOLS_DIR / "few_shot_pool_t32.json",
    "t33": POOLS_DIR / "few_shot_pool_t33.json",
}

def _pool_is_stale(task, cached):
    """Detecta si el pool cacheado fue generado con la version vieja del builder
    (sin lang, sin co-ocurrencia, con NO en T3.2/T3.3)."""
    if task == "t32":
        if any(ex.get("label") == "NO" for ex in cached): return True
    if task == "t33":
        if any(ex.get("labels") == ["NO"] for ex in cached): return True
        # Pool antiguo no incluia los 2 extras de co-ocurrencia
        expected = 5 * N_PER_CLASS_T33 + N_T33_COOCCUR
        if len(cached) != expected: return True
    if not cached or "lang" not in cached[0]: return True
    return False

def _build_default_pools(seed=None):
    return {
        "t31": select_few_shot_pool_t31(df_train, raw_train, seed=seed),
        "t32": select_few_shot_pool_t32(df_train, raw_train, seed=seed),
        "t33": select_few_shot_pool_t33(df_train, raw_train, seed=seed),
    }

POOLS = {}
for task in ["t31", "t32", "t33"]:
    p = POOL_PATHS[task]
    if p.exists():
        cached = json.load(open(p, "r", encoding="utf-8"))
        if _pool_is_stale(task, cached):
            print(f"[{task}] pool obsoleto, regenerando...")
            POOLS[task] = _build_default_pools(seed=None)[task]
            with open(p, "w", encoding="utf-8") as f:
                json.dump(POOLS[task], f, ensure_ascii=False, indent=2)
        else:
            POOLS[task] = cached
            print(f"[{task}] pool cargado de cache ({len(POOLS[task])} ej.) - {p.name}")
    else:
        POOLS[task] = _build_default_pools(seed=None)[task]
        with open(p, "w", encoding="utf-8") as f:
            json.dump(POOLS[task], f, ensure_ascii=False, indent=2)
        print(f"[{task}] pool generado ({len(POOLS[task])} ej.) -> {p.name}")

POOL_IDS = set()
for task in ["t31", "t32", "t33"]:
    for ex in POOLS[task]: POOL_IDS.add(str(ex["id"]))
print(f"Total IDs unicos en pools: {len(POOL_IDS)}")

for task in ["t31", "t32", "t33"]:
    print()
    print(f"  --- pool {task} (n={len(POOLS[task])}) ---")
    for ex in POOLS[task]:
        lbl = ex.get("label") or ex.get("labels")
        lang = ex.get("lang", "?")
        print(f"    id={str(ex['id']):>8}  cons={ex['consensus']:.2f}  lang={lang}  label={lbl}")


[t31] pool cargado de cache (6 ej.) - few_shot_pool_t31.json
[t32] pool cargado de cache (6 ej.) - few_shot_pool_t32.json
[t33] pool cargado de cache (12 ej.) - few_shot_pool_t33.json
Total IDs unicos en pools: 19

  --- pool t31 (n=6) ---
    id=  120002  cons=1.00  lang=es  label=YES
    id=  120005  cons=1.00  lang=es  label=YES
    id=  120001  cons=0.67  lang=es  label=YES
    id=  120003  cons=1.00  lang=es  label=NO
    id=  120006  cons=1.00  lang=es  label=NO
    id=  120033  cons=0.67  lang=es  label=NO

  --- pool t32 (n=6) ---
    id=  120002  cons=1.00  lang=es  label=DIRECT
    id=  120005  cons=1.00  lang=es  label=DIRECT
    id=  120001  cons=0.67  lang=es  label=DIRECT
    id=  120055  cons=1.00  lang=es  label=JUDGEMENTAL
    id=  120056  cons=1.00  lang=es  label=JUDGEMENTAL
    id=  120058  cons=0.67  lang=es  label=JUDGEMENTAL

  --- pool t33 (n=12) ---
    id=  120065  cons=1.00  lang=es  label=['IDEOLOGICAL-INEQUALITY']
    id=  120066  cons=1.00  lang=es  label=

## 4 - Plantillas de prompt por subtask

Las definiciones se mantienen literales del lab guidelines V0.5 pag. 2-4. Sustituimos "meme" por "video" y reforzamos en T3.2/T3.3 que el video ya esta marcado como sexista (consistencia con la jerarquia).

In [9]:
# === 4.1 - Construccion de mensajes (chat-template comun Qwen-VL / Gemma-4) =

SYSTEM_T31 = """You are an expert annotator identifying sexism in bilingual (English/Spanish) short-form videos.

# TASK
Decide whether a video is SEXIST (YES) or NOT SEXIST (NO).

# DEFINITION
A video is SEXIST if it contains sexist expressions or behaviours — that is, it is sexist itself, depicts a sexist situation, OR critically engages with / denounces sexism. The last clause is part of the definition: counter-speech that treats sexism as the subject is YES, not NO.

However, a mere gendered relationship scenario is not enough. Boyfriend/girlfriend, husband/wife, dating, attractiveness, jealousy, parenting, friendship, or family content is YES only if it invokes gender-based hierarchy, stereotype, objectification, restriction, harassment, violence, anti-feminism, or explicit critical engagement with sexism.

# YES — assign when ANY of these apply

1. Overt sexism. Slurs, insults, harassment, dehumanizing depictions targeting women or LGBTQ+ people on the basis of gender or gender identity (spoken, written, or visually staged).

2. Stereotypes used as the joke's premise. "Women drivers", "back to the kitchen", "men can't cry", scenario-format videos like "POV: my girlfriend trying to read a map". The humor only lands because of an essentialist gender claim.

3. Benevolent sexism. Apparently positive framings tied to traditional roles or restrictive virtues — content celebrating submissive domesticity, "high-value woman" rules, "real women cook for their man", "good girls don't…". One of the most common false negatives. Do not skip it because the tone sounds aspirational.

4. Anti-feminism / men-as-victims. Discrediting feminism, claiming equality is already achieved, framing men as oppressed by women / law / "modern dating", mocking #MeToo, content with redpill / MGTOW / "alpha male" anti-feminist messaging.

5. Sexualization or objectification by third parties. "Rate her", "would/wouldn't", body-shaming, hypersexualizing framings where a woman is treated as an object. A woman self-presenting is not automatically sexist; sexism arises when the framing or audience is positioned to objectify.

6. Sexual-violence framing. Jokes, skits, or "tips" about rape, harassment, coercion, or stalking, even when packaged as humor or relationship advice ("how to convince her").

7. Critical engagement with sexism / counter-speech. Storytimes denouncing harassment, parodies of misogynistic creators, response videos calling out sexists, feminist activism, "let me tell you what this guy did" formats. YES because the video deals with sexism. Requires observable critical engagement in at least one channel: transcript, on-screen text, visual performance, editing, reply context, facial expression, tone, or narrative framing — not mere mention of a gendered topic.

8. Sexist irony / sarcasm whose punchline reinforces a stereotype. Irony does not flip the label here. A skit "playing" a clueless woman to get laughs at women's expense is YES.

9. Performance of sexist audio. Selecting and performing (lip-syncing, miming, repeating) an audio whose lyrics or content are sexist counts as YES at this task, regardless of whether the creator endorses or critiques it. Whose side the creator is on is decided downstream, not here.

# NO — assign when

10. The video is unrelated to gender. Cooking, dance challenges, sports, pets, programming, generic comedy where gender is not the hinge.

11. The video features a woman or man, but the content is not about their gender. A female chef demonstrating a recipe; a female athlete celebrating a win; a male nurse explaining a procedure. If you can swap the creator's gender and the content lands identically, it is almost certainly NO.

12. The video references women or men neutrally without invoking inferiority, hierarchy, sexualization, restriction, essentialist traits, harassment, violence, anti-feminism, or explicit critical engagement with sexism.

13. The video contains a relationship, dating, couple, family, or attractiveness scenario, but gender is incidental and no sexist stereotype, hierarchy, restriction, objectification, harassment, violence, or critique of sexism is present.

# DECISION HEURISTICS

- ARBITER QUESTION: "Does the message require a specific gender to land because it relies on a stereotype, hierarchy, restriction, objectification, harassment, violence, anti-feminism, or explicit critique of sexism?" Yes → lean YES. If swapping gender breaks nothing or only changes surface details → lean NO.

- TRANSCRIPT vs VISUAL CONFLICT: when audio and framing contradict, identify which channel carries the communicative intent. A woman saying "I'm so glad my husband decides everything" while smirking and rolling her eyes is critical engagement (YES, counter-speech). A woman saying "of course women can do anything" while overlay text says "except parking" is direct stereotyping (YES).

- SCENARIO / POV FORMAT: scenario openers ("POV:", "When she…", "Every man at…") invite the viewer to inhabit a perspective. The label depends on whose perspective and against whom. Mocking female suspicion → YES. Mocking male reaction in feminist contexts → YES. Voicing a woman's experience of workplace sexism critically → YES. Generic relationship comedy without a gender-based stereotype or hierarchy → NO.

- REPLY / RESPONSE FORMAT: amplifying or endorsing the original sexist content → YES. Ridiculing or refuting it → YES (counter-speech). Either way YES.

- SHORT / FRAGMENTARY: very short clips may not have enough signal. If the gender hinge is not clearly visible and stereotypes / attacks are not visually unambiguous, lean NO.

- BILINGUAL: same criteria for English and Spanish.

# CALIBRATION
Borderline cases lean YES only when the gendered framing carries a stereotype, hierarchy, objectification, restriction, harassment, violence, anti-feminism, or explicit critical engagement with sexism. Cases where gender is incidental to the content lean NO. Pick the label more strongly supported by the actual signals; do not force certainty when the signals do not warrant it.

# OUTPUT
You must classify every video. No refusal, no hedging.
Answer with EXACTLY one token: YES or NO.
No punctuation, no explanation, no quotes, no extra words."""

SYSTEM_T32 = """You are an expert annotator classifying source intention in bilingual (English/Spanish) short-form videos already determined to be sexist.

# TASK
The video has been classified as sexist by an upstream system. Assign source intention: DIRECT or JUDGEMENTAL. There is no NO option — the upstream classifier already filtered non-sexist content out.

# THE CORE QUESTION
WHOSE SIDE IS THE CREATOR ON?

  DIRECT      → the creator endorses, perpetuates, or jokes from inside the
                sexist worldview. Women or feminism are the target. The video
                reinforces a sexist stance.

  JUDGEMENTAL → the creator condemns, denounces, parodies, or testifies against
                sexism. Sexists, harassers, misogynistic norms, or discriminatory
                systems are the target. The video critically engages with sexist
                behaviour, beliefs, structures, or experiences.

# DIRECT — operational signals

D1. Punchline targets women or feminism. Comedy or scorn aimed at women's competence, behaviour, morality, sexuality, activism, or appearance; anti-feminist mockery.

D2. Tradwife / "high-value woman" content presented aspirationally. Celebrating submissive domesticity, "feminine energy" rules prescribing restrictive roles. Benevolent sexism is almost always DIRECT when the creator endorses the restrictive norm.

D3. Redpill / MGTOW / "alpha male" content. "Modern women are entitled", "passport bros", "women hit the wall at 30", coaching that positions men as victims or rightful dominators.

D4. Sexist sketches aimed downward. A skit where a woman is portrayed as irrational, gold-digging, dramatic, promiscuous, manipulative, or incompetent and the audience is invited to laugh AT her.

D5. Objectification framings. "Rate her", "would/wouldn't", body commentary as the content, hypersexualized framings of women as available objects.

D6. Sexual-violence "humor". Coercion or assault framed as playful, "she'll thank me later", "convince her" content.

D7. Performing sexist audio sincerely. Creator chose the audio and performs it without ironic distance — they are amplifying the original message.

D8. Misogynistic monologues delivered as truth-telling, dating advice, social commentary, or self-improvement content.

# JUDGEMENTAL — operational signals (require observable critique)

J1. Storytime denouncing harassment or sexism. The creator shares an experience of sexism with disapproving, angry, hurt, critical, or reflective framing.

J2. Parody of sexists, not of women. The exaggeration ridicules the sexist worldview itself.

J3. Reply / response calling out sexist content. The creator responds to a misogynistic clip to refute or annotate it critically. The original is shown as the bad example.

J4. Feminist activism / counter-speech. Pro-equality monologues, denouncing double standards, supporting victims, calling out systemic sexism.

J5. Reframing sexist tropes critically. Opens with a sexist setup and visually, verbally, or narratively subverts it, making the trope look ridiculous.

J6. Observable markers of disapproval. Eye-rolling, mock-incredulous tone, sarcastic repetition of a sexist statement followed by counter-evidence, pained or angry expressions while reciting received insults, captions such as "this guy actually thinks this", "men really say this", "not okay", "stop normalizing this".

JUDGEMENTAL requires observable critical engagement in at least one channel: transcript, on-screen text, visual performance, editing, reply context, facial expression, tone, or narrative framing. Mere irony, ambiguity, or the presence of a woman creator is not enough.

# HARD CASES — DECISION RULES

- IRONY / SARCASM ALONE IS NOT JUDGEMENTAL. Ask: when the irony lands, who is being ridiculed? Women / feminism → DIRECT. Sexists / misogynistic norms → JUDGEMENTAL. Sexist irony is common in comedy formats; counter-speech irony is the exception. If the target of irony is unclear, weight other signals (disapproval markers, on-screen text, who is defeated in the punchline, reply context, narrative resolution).

- SKIT WITH A SEXIST CHARACTER. Look at the punchline. Sexist character shown as ridiculous, defeated, contradicted, exposed, or morally wrong → JUDGEMENTAL. Sexist character shown as right, vindicated, desirable, funny because "it is true", or expressing "the truth nobody says" → DIRECT.

- PERFORMING SOMEONE ELSE'S AUDIO. The performance alone tells you nothing about intent. The upstream task has already established the audio is sexist; here, what matters is the surrounding context. Sincere performance → DIRECT. Performance with mocking face + caption like "this guy actually thinks this" → JUDGEMENTAL.

- SCENARIO / POV FORMAT. "When she…" mocking women → DIRECT. Giving voice to victims of a sexist scenario → JUDGEMENTAL. Showing the sexist scenario as normal, funny, or justified → DIRECT. Showing the sexist scenario as unfair, absurd, traumatic, or worth criticizing → JUDGEMENTAL.

- TRANSCRIPT vs VISUAL CONTRADICTION. When sexist words pair with disapproving facial expressions, reply context, or overlay text that calls them out, the video is JUDGEMENTAL counter-speech. Trust the channel that carries the punchline.

- "JUST JOKING" disclaimer does not turn DIRECT into JUDGEMENTAL. The video still endorses the framing if its punchline targets women or feminism.

- BILINGUAL irony markers. Spanish: "claro, claro…", "obvio…", "ay sí, sí", "nada que ver", "sí, seguro", "cómo no". English: "/s", "amirite", "totally", "of course", "yeah right", "sure".

# CLASS DISTRIBUTION — weak expectation only
DIRECT may be more frequent than JUDGEMENTAL in comparable short-form sexism corpora, but this is only a weak expectation. Never use class distribution to override evidence from the video. Use it only when the signals are genuinely unresolved after applying all rules.

# IF SIGNALS ARE GENUINELY MIXED
Pick the side that the predominant signal supports. JUDGEMENTAL requires observable critical engagement; if you cannot identify any critique in transcript, on-screen text, visual performance, editing, reply context, facial expression, tone, or narrative framing, DIRECT is more likely. Do not force certainty when the signals do not warrant it.

# OUTPUT
You must classify every video. Refusing or hedging is not an option.
Answer with EXACTLY one of: DIRECT or JUDGEMENTAL.
No punctuation, no explanation, no quotes, no extra words."""

SYSTEM_T33 = """You are an expert annotator categorizing types of sexism in bilingual (English/Spanish) short-form videos already determined to be sexist.

# TASK
The video has been classified as sexist by an upstream system. This is a MULTI-LABEL classification: assign one OR several of the five categories below. There is no NO option.

# LABEL SPACE (output only these labels, exactly)
  IDEOLOGICAL-INEQUALITY
  STEREOTYPING-DOMINANCE
  OBJECTIFICATION
  SEXUAL-VIOLENCE
  MISOGYNY-NON-SEXUAL-VIOLENCE

# HARD RULES

H1. Categories often co-occur. A redpill rant with stereotyping language and dehumanizing slurs gets at least three. Do not force single-label.

H2. JUDGEMENTAL framings still get categories. A storytime denouncing rape culture is tagged with SEXUAL-VIOLENCE — this task asks WHICH type of sexism appears in the video, regardless of the creator's stance toward it.

H3. Use every available channel (frames, transcript, on-screen text, audio context). If a channel is missing, judge from what remains.

H4. For each output label, there must be an explicit signal supporting that exact category. Do not infer additional labels from general sexist tone unless the category definition is directly satisfied.

H5. Do not pad the output with plausible but unsupported categories. Multilabel does not mean maximal-label: output all and only the categories clearly supported by the video.

# CATEGORY DEFINITIONS

──────────────────────────────────────────────────────────────────────────────
IDEOLOGICAL-INEQUALITY — discredits feminism, denies gender inequality exists,
or frames men as victims of gender-based oppression.
──────────────────────────────────────────────────────────────────────────────
Apply when the video:
  - Mocks feminism, feminists, or feminist activism.
  - Claims equality is "already achieved" or that women are now privileged.
  - Frames men as the real victims (family courts, false accusations,
    dating market, "gender ideology").
  - Belongs to redpill / MGTOW / "alpha male" / "passport bro" ecosystems
    with anti-feminist messaging.
Cues: "modern women are…" rants, manosphere podcast clip responses,
"this is what feminism did to society" hot-takes, mocking impressions of
"online feminism".

Do not use IDEOLOGICAL-INEQUALITY merely because the video is sexist. Use it only when feminism, gender equality, men's victimhood, redpill/manosphere ideology, anti-feminist politics, or structural gender politics are part of the message.

──────────────────────────────────────────────────────────────────────────────
STEREOTYPING-DOMINANCE — false ideas about women's traits or roles, or claims
of male superiority.
──────────────────────────────────────────────────────────────────────────────
Apply when the video:
  - Reinforces traditional gender roles (tradwife content, "real women cook
    for their man", "a woman's place is at home").
  - Mocks women's competence in coded domains (driving, sports, programming,
    finance, leadership, parking, directions).
  - Asserts essentialist traits ("women are emotional", "men are logical",
    "women can't read maps").
  - Claims male superiority in any domain.
  - Includes benevolent stereotypes ("a real lady is gentle", "high-value
    woman" prescriptions). Benevolent is still stereotyping.

──────────────────────────────────────────────────────────────────────────────
OBJECTIFICATION — presents women as objects detached from personhood, or
prescribes physical qualities they must meet.
──────────────────────────────────────────────────────────────────────────────
Apply when the video:
  - Reduces a woman to her body / body parts.
  - "Rate her" / "would/wouldn't" / "1-10" formats.
  - Imposes beauty standards ("real women have curves" / "real women are
    skinny" — both qualify), body-shaming.
  - Treats female bodies as decoration, prize, trophy, reward, or commodity.
  - Hypersexualizes female anatomy WITHOUT a coercive / non-consensual frame.

A woman self-presenting, dancing, dressing attractively, or showing her own body is not automatically OBJECTIFICATION. The category applies when the framing, captions, audio, camera work, commentary, or audience position reduces her to an object, rating, commodity, body part, or sexual availability.

──────────────────────────────────────────────────────────────────────────────
SEXUAL-VIOLENCE — sexual suggestions, coercion, harassment, rape, assault.
──────────────────────────────────────────────────────────────────────────────
Apply when the video:
  - Jokes about rape, sexual assault, or coercion (even "ironically").
  - Frames non-consensual scenarios as humorous or as a "tip".
  - Depicts sexual harassment as positive, normal, or funny.
  - Includes "just don't say no" / "she'll thank me later" / "convince her"
    framings as relationship/dating advice.
  - Storytimes recounting sexual assault from the victim's side ALSO qualify
    (per H2).
ES cues: "se hace la difícil", "al final cae", "insiste y verás",
coerción presentada como seducción.
EN cues: "convince her", "she's playing hard to get", "wear her down".

──────────────────────────────────────────────────────────────────────────────
MISOGYNY-NON-SEXUAL-VIOLENCE — hatred toward women, plus non-sexual physical
or verbal violence toward women.
──────────────────────────────────────────────────────────────────────────────
Apply when the video:
  - Expresses hatred or disgust toward women as a class ("women are the
    worst", "femoids", incel terminology).
  - Jokes about hitting, beating, killing, or otherwise physically harming
    women in non-sexual ways. Domestic violence "jokes".
  - Includes slurs targeting women collectively.
  - Frames violence against women as deserved or funny.
  - Incel / blackpill content with explicit hate framing.

Do not use MISOGYNY-NON-SEXUAL-VIOLENCE for ordinary stereotypes, mockery, dating complaints, or objectification unless there is explicit hatred, disgust, slurs, dehumanization, or non-sexual physical/verbal violence toward women.

# DISAMBIGUATION

R1. STEREOTYPING-DOMINANCE  vs  IDEOLOGICAL-INEQUALITY
    - STEREOTYPING is about traits and roles ("women belong in the kitchen",
      tradwife prescriptions, "women can't drive").
    - IDEOLOGICAL is about the political / ideological frame ("feminism is
      cancer", "equality already exists", redpill ideology).
    - They co-occur frequently. Tag both when both are explicitly supported.

R2. OBJECTIFICATION  vs  SEXUAL-VIOLENCE
    - OBJECTIFICATION = commodification / beautification framing without a
      coercive element.
    - SEXUAL-VIOLENCE = coercive, non-consensual, harassment, or assault framing
      (even as a joke).
    - "Rate her" → OBJECTIFICATION. "How to convince her" → SEXUAL-VIOLENCE.
      Hypersexualized + coercive → BOTH.

R3. MISOGYNY-NON-SEXUAL-VIOLENCE  vs  STEREOTYPING-DOMINANCE
    - MISOGYNY-NSV = hatred or non-sexual violence (slurs, dehumanization,
      hitting jokes).
    - STEREOTYPING = roles and traits asserted as fact, without pure hate.
    - "Get back in the kitchen" alone → STEREOTYPING. With violent imagery,
      slurs, dehumanization, or threat → STEREOTYPING + MISOGYNY-NSV.

R4. IDEOLOGICAL-INEQUALITY  vs  MISOGYNY-NON-SEXUAL-VIOLENCE
    - "Feminists are stupid" arguing against the movement → IDEOLOGICAL.
    - "I hate women" without an ideological frame → MISOGYNY-NSV.
    - Incel monologues mixing both → BOTH.

R5. STEREOTYPING-DOMINANCE  vs  OBJECTIFICATION
    - STEREOTYPING = women are assigned traits, roles, duties, or inferior capacities.
    - OBJECTIFICATION = women are evaluated as bodies, objects, trophies, or sexual commodities.
    - "Women should cook for men" → STEREOTYPING.
    - "She is a 4/10" → OBJECTIFICATION.
    - "A high-value woman must be beautiful, quiet, and serve her man" → STEREOTYPING + OBJECTIFICATION.

# COMMON COMBINATIONS (illustrative)
  STEREOTYPING-DOMINANCE
  OBJECTIFICATION
  STEREOTYPING-DOMINANCE, IDEOLOGICAL-INEQUALITY
  STEREOTYPING-DOMINANCE, OBJECTIFICATION
  OBJECTIFICATION, SEXUAL-VIOLENCE
  IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-VIOLENCE
  IDEOLOGICAL-INEQUALITY, STEREOTYPING-DOMINANCE, MISOGYNY-NON-SEXUAL-VIOLENCE

# IF SIGNALS ARE GENUINELY MIXED
Apply each category whose definition is clearly supported by at least one signal in the video. If only one is clearly supported, output a single category. If multiple are clearly supported, output multiple. Do not add categories by association, ideology, or background knowledge unless the video itself supports them.

# OUTPUT
You must classify every video. Refusing or hedging is not an option.

Output a comma-separated list of one or more categories using EXACTLY the label names above (uppercase, hyphenated, no abbreviations, no aliases).
Examples of valid outputs:
  STEREOTYPING-DOMINANCE
  OBJECTIFICATION, SEXUAL-VIOLENCE
  IDEOLOGICAL-INEQUALITY, STEREOTYPING-DOMINANCE, MISOGYNY-NON-SEXUAL-VIOLENCE

No explanations, prefixes, quotes, trailing punctuation, or extra words. Just the labels."""


USER_QUESTION = {
    "t31": "Looking at the video and the transcribed text together, is this video sexist? Answer YES or NO.",
    "t32": "This video is sexist. Looking at the video and the transcribed text together, what is the source intention? Answer DIRECT or JUDGEMENTAL.",
    "t33": "This video is sexist. Looking at the video and the transcribed text together, which sexism categories apply? Answer with one or more categories from the label space.",
}

# Parametros de muestreo de frames del video (solo Qwen-VL los usa explicitamente).
# Reduce si tienes OOM. En Gemma el processor controla la resolucion via su
# visual_token_budget (configurable mediante AutoProcessor.from_pretrained args).
VIDEO_MAX_PIXELS = 480 * 640    # ~307K px, ~512 visual tokens (Qwen)
VIDEO_FPS        = 0.5          # ~40 frames en un video de 80s (Qwen)

def _id_to_videopath(vid):
    return df.loc[df["id_EXIST"] == str(vid), "video_path"].iloc[0]

def _id_to_ocr(vid):
    s = df.loc[df["id_EXIST"] == str(vid), "text_ocr"].iloc[0]
    return (s or "").strip()

# === Rationale opcional en los shots ========================================
# Cuando USE_FEW_SHOT_RATIONALE = True, cada shot del few-shot pool va seguido
# de un guion + rule_id + rationale corta (UNA frase) extraida de las reglas
# del system prompt. Mapping rule_id -> texto corto:
RATIONALE_T31 = {
    "1":  "Overt sexism: slurs or dehumanizing depictions targeting women or LGBTQ+.",
    "2":  "Gender stereotypes used as the joke's premise.",
    "3":  "Benevolent sexism: positive-sounding framings tying women to restrictive roles.",
    "4":  "Anti-feminism or men-as-victims framing.",
    "5":  "Sexualization or objectification of women.",
    "6":  "Sexual-violence framing, including coercion or harassment as humor.",
    "7":  "Counter-speech: video critically engages with sexism.",
    "8":  "Ironic delivery whose punchline reinforces a stereotype.",
    "9":  "Lip-sync to sexist audio that the creator amplifies.",
    "10": "Unrelated to gender (cooking, sports, pets, etc.).",
    "11": "Features a woman but content is not about her gender.",
    "12": "Neutral reference to women or men with no hierarchy or stereotype.",
}
RATIONALE_T32 = {
    "D1": "Punchline targets women or feminism.",
    "D2": "Tradwife / high-value-woman content presented as aspirational.",
    "D3": "Redpill or alpha-male framing where men are positioned as victims.",
    "D4": "Sketch aimed downward — woman portrayed as incompetent for laughs.",
    "D5": "Sexualization or rate-her framing of female bodies.",
    "D6": "Sexual-violence humor framing coercion as playful.",
    "D7": "Lip-syncs sexist audio sincerely, amplifying its message.",
    "D8": "Misogynistic monologue delivered as advice or truth-telling.",
    "J1": "Storytime denouncing harassment with critical tone.",
    "J2": "Parody whose target is the sexist worldview, not women.",
    "J3": "Stitch or duet calling out sexist content.",
    "J4": "Feminist activism or pro-equality counter-speech.",
    "J5": "Sexist trope reframed and visually subverted.",
    "J6": "Visible markers of disapproval toward sexism received.",
}
RATIONALE_T33 = {
    "IDEOLOGICAL-INEQUALITY":       "Anti-feminist or men-as-victims ideological framing.",
    "STEREOTYPING-DOMINANCE":       "Reinforces traditional gender roles or essentialist traits.",
    "OBJECTIFICATION":              "Reduces a woman to her body or imposes beauty standards.",
    "SEXUAL-VIOLENCE":              "Sexual coercion, assault or harassment framing.",
    "MISOGYNY-NON-SEXUAL-VIOLENCE": "Hatred toward women or non-sexual violence framing.",
}
DEFAULT_RULE_T31 = {"YES": "2",  "NO": "10"}
DEFAULT_RULE_T32 = {"DIRECT": "D4", "JUDGEMENTAL": "J1"}

def _format_label_for_assistant(task, ex):
    if task == "t31":
        label = ex["label"]
        if not USE_FEW_SHOT_RATIONALE: return label
        rid = DEFAULT_RULE_T31[label]
        return f"{label} - {rid}: {RATIONALE_T31[rid]}"
    if task == "t32":
        label = ex["label"]
        if not USE_FEW_SHOT_RATIONALE: return label
        rid = DEFAULT_RULE_T32[label]
        return f"{label} - {rid}: {RATIONALE_T32[rid]}"
    if task == "t33":
        labels_str = ", ".join(ex["labels"])
        if not USE_FEW_SHOT_RATIONALE: return labels_str
        primary = ex["labels"][0]
        return f"{labels_str} - {primary}: {RATIONALE_T33[primary]}"
    raise ValueError(task)

def _video_content_block(video_path):
    """Bloque de contenido tipo 'video' segun el backend activo."""
    if BACKEND == "qwen":
        return {"type": "video", "video": str(video_path),
                "max_pixels": VIDEO_MAX_PIXELS, "fps": VIDEO_FPS}
    # Gemma: el processor lee el video con su default sampling (1 fps, max 60s).
    return {"type": "video", "video": str(video_path)}

def _user_block(video_path, ocr_text, question):
    body = f"This video has the transcribed text: \"{ocr_text}\"\n{question}" if ocr_text \
        else f"(The video has no extracted text.)\n{question}"
    # Modality-first: la guia de Gemma recomienda video ANTES del texto;
    # Qwen-VL acepta este mismo orden sin penalizacion.
    return {
        "role": "user",
        "content": [
            _video_content_block(video_path),
            {"type": "text", "text": body},
        ],
    }

def build_messages(task, query_video_path, query_ocr, pool_examples):
    sys_prompt = {"t31": SYSTEM_T31, "t32": SYSTEM_T32, "t33": SYSTEM_T33}[task]
    msgs = [{"role": "system", "content": sys_prompt}]
    q    = USER_QUESTION[task]
    for ex in pool_examples:
        msgs.append(_user_block(_id_to_videopath(ex["id"]), _id_to_ocr(ex["id"]), q))
        msgs.append({"role": "assistant",
                     "content": _format_label_for_assistant(task, ex)})
    msgs.append(_user_block(query_video_path, query_ocr, q))
    return msgs

# Smoke test ligero (no llama al modelo)
_query_pool = df_test if RUN_MODE == "test" else df_train
_test_query = _query_pool[~_query_pool["id_EXIST"].isin(POOL_IDS)].iloc[0]
_test_msgs = build_messages("t31", _test_query["video_path"], _test_query["text_ocr"], POOLS["t31"])
print(f"build_messages OK | n_messages = {len(_test_msgs)}  | query split = {_test_query['split']}  | backend = {BACKEND}")


build_messages OK | n_messages = 14  | query split = test  | backend = gemma


## 5 - Funcion de inferencia + parser robusto

Decoding greedy (do_sample=False). Parser tolera variantes (mayus/minus, ordenacion). Los fallbacks se loguean en `parse_fallbacks.log`.

In [10]:
# === 5.1 - Parser de respuestas (con stripping de razonamiento) =============
FALLBACK_LOG = LOGS_DIR / "parse_fallbacks.log"
_fallback_count = {"t31": 0, "t32": 0, "t33": 0}
_total_count    = {"t31": 0, "t32": 0, "t33": 0}

def _log_fallback(task, vid, raw_resp, used, exc_type=None):
    rec = {"task": task, "id": vid, "resp": raw_resp[:500], "fallback": used}
    if exc_type is not None:
        rec["exc_type"] = exc_type
    with open(FALLBACK_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

# El modelo Thinking emite "<think>...razonamiento...</think>respuesta_final".
# El chat_template del modelo abre <think> implicitamente, asi que en la salida
# decoded suele aparecer SOLO el cierre "</think>". Cortamos por ese cierre.
_THINK_CLOSE_RE = re.compile(r"</think\s*>", re.IGNORECASE)

def _strip_thinking(raw_resp):
    """Quita el bloque de razonamiento de un Thinking model. Devuelve la
    respuesta limpia. Si no hay </think>, devuelve la cadena original."""
    if not raw_resp:
        return raw_resp
    m = _THINK_CLOSE_RE.search(raw_resp)
    if m:
        return raw_resp[m.end():].strip()
    # Algunas variantes incluyen <think> abierto explicito.
    if "<think>" in raw_resp.lower():
        # No hay cierre -> el modelo se quedo razonando; nos quedamos con lo que haya
        # tras el ultimo </think> hipotetico (no hay): caemos al fallback del parser.
        return raw_resp
    return raw_resp

_VALID_T32 = {"DIRECT", "JUDGEMENTAL"}      # NO se filtra por jerarquia
_VALID_T33 = set(SEXISM_CATS)               # idem
_T32_SYNONYMS = {"DIRECTLY": "DIRECT", "JUDGMENT": "JUDGEMENTAL",
                  "JUDGMENTAL": "JUDGEMENTAL", "JUDGE": "JUDGEMENTAL"}

def parse_response(resp, task, vid="?"):
    _total_count[task] += 1
    # Solo se llama a _strip_thinking si el backend es Thinking.
    cleaned = _strip_thinking(resp) if IS_THINKING else (resp or "")
    s = (cleaned or "").strip().upper()
    s = re.sub(r"^[\s\W_]*ANSWER\s*[:=]\s*", "", s)
    s = re.sub(r"^[\s\W_]*THE\s+ANSWER\s+IS\s+", "", s)
    if task == "t31":
        if re.search(r"\bNO\b", s) and not re.search(r"\bYES\b", s): return "NO"
        if re.search(r"\bYES\b", s) and not re.search(r"\bNO\b", s): return "YES"
        first = re.split(r"[\s,.;:!?]+", s, maxsplit=1)[0] if s else ""
        if first in {"YES", "Y", "TRUE", "1"}: return "YES"
        if first in {"NO", "N", "FALSE", "0"}: return "NO"
        _fallback_count[task] += 1
        _log_fallback(task, vid, resp, "NO")
        return "NO"
    if task == "t32":
        for tok in re.split(r"[\s,.;:!?]+", s):
            tok = _T32_SYNONYMS.get(tok, tok)
            if tok in _VALID_T32: return tok
        _fallback_count[task] += 1
        _log_fallback(task, vid, resp, "DIRECT")
        return "DIRECT"   # fallback DIRECT (mas frecuente que JUDG)
    if task == "t33":
        s_clean = re.sub(r"[\.\;\!\?\n]", ",", s)
        toks = [t.strip() for t in s_clean.split(",") if t.strip()]
        cats = []
        for tok in toks:
            tok = tok.replace("_", "-").replace(" ", "-")
            if tok in _VALID_T33: cats.append(tok)
            else:
                for full in SEXISM_CATS:
                    if full.startswith(tok) or tok in full.split("-"):
                        cats.append(full); break
        cats = list(dict.fromkeys(cats))
        if cats: return cats
        _fallback_count[task] += 1
        # Si la jerarquia llego aqui, el video YA es sexista. Fallback a la
        # categoria mas frecuente: STEREOTYPING-DOMINANCE.
        _log_fallback(task, vid, resp, str([T33_FALLBACK_CATEGORY]))
        return [T33_FALLBACK_CATEGORY]
    raise ValueError(task)


In [11]:
# === 5.2 - predict_video (forward + decode + parse, dispatch por BACKEND) ===
# Cada backend prepara los inputs de forma distinta:
#   - qwen : usamos qwen_vl_utils.process_vision_info para extraer frames y
#            llamamos a processor(text=..., images=..., videos=...)
#   - gemma: el processor.apply_chat_template con tokenize=True ya devuelve
#            input_ids + pixel_values listos. No necesitamos qwen_vl_utils.
#
# NOTA SOBRE THINKING: tanto Qwen3.5 como Gemma-4 pueden razonar antes de
# emitir el label. Si max_new_tokens es bajo, la respuesta queda truncada en
# el razonamiento y el parser cae al fallback. Por eso:
#   1) max_new_tokens grande (configurado en BCFG)
#   2) strip_thinking() quita <think>...</think> antes del parser
#   3) enable_thinking=False en apply_chat_template para Qwen (cuando soportado)
if BACKEND == "qwen":
    try:
        from qwen_vl_utils import process_vision_info
    except ImportError:
        import subprocess as _sp, sys as _sys, importlib as _il
        print("Instalando qwen-vl-utils[decord] on-the-fly...")
        _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                         "qwen-vl-utils[decord]"])
        _il.invalidate_caches()
        from qwen_vl_utils import process_vision_info

MAX_NEW_TOKENS = BCFG["max_new_tokens"]

_THINK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)
_THINK_OPEN_ONLY_RE = re.compile(r"^.*?</think>", flags=re.DOTALL | re.IGNORECASE)

def strip_thinking(s: str) -> str:
    """Quita bloques <think>...</think> y, si hay </think> sin <think> al
    inicio (raw truncado al principio), descarta todo lo previo."""
    if not s:
        return s
    s2 = _THINK_RE.sub("", s)
    if "</think>" in s2.lower():
        s2 = _THINK_OPEN_ONLY_RE.sub("", s2)
    return s2.strip()

def _prepare_inputs_qwen(messages):
    # Qwen3.5 soporta enable_thinking=False; versiones antiguas no.
    try:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except TypeError:
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)
    return inputs

def _prepare_inputs_gemma(messages):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)
    return inputs

_PREPARE_INPUTS = _prepare_inputs_qwen if BACKEND == "qwen" else _prepare_inputs_gemma

@torch.no_grad()
def predict_video(vid_id, video_path, ocr_text, task, pool_examples,
                   return_raw=False):
    messages = build_messages(task, video_path, ocr_text, pool_examples)
    inputs = _PREPARE_INPUTS(messages)

    out_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS[task],
        do_sample=False,
        temperature=None,
        top_p=None,
    )
    gen = out_ids[:, inputs.input_ids.shape[1]:]
    raw_resp = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
    # Strip thinking blocks ANTES del parser. Si no hay <think>, no-op.
    clean_resp = strip_thinking(raw_resp)
    parsed = parse_response(clean_resp, task, vid_id)
    if return_raw:
        return parsed, raw_resp   # devolvemos raw original (con thinking) para debug
    return parsed

# Smoke test sobre 1 video (T3.1)
print(f"--- Smoke test predict_video (T3.1) | BACKEND={BACKEND} | RUN_MODE={RUN_MODE} ---")
print(f"    MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")
if RUN_MODE == "train_eval":
    eval_df = df_train[~df_train["id_EXIST"].isin(POOL_IDS) & df_train["t31_hard"].notna()]
    smoke_ids = []
    for cls in (1, 0):
        sub = eval_df[eval_df["t31_hard"] == cls].head(1)
        if len(sub): smoke_ids.append(sub.iloc[0]["id_EXIST"])
    for vid in smoke_ids[:1]:
        row = df.loc[df["id_EXIST"] == vid].iloc[0]
        pred, rraw = predict_video(vid, row["video_path"], row["text_ocr"],
                                    "t31", POOLS["t31"], return_raw=True)
        gt = T31_INT_TO_STR[int(row["t31_hard"])]
        print(f"  id={vid}  GT={gt}  PRED={pred}")
        print(f"  RAW (truncated)={rraw[:300]!r}")
else:
    smoke_ids = df_test.head(1)["id_EXIST"].tolist()
    for vid in smoke_ids:
        row = df.loc[df["id_EXIST"] == vid].iloc[0]
        pred, rraw = predict_video(vid, row["video_path"], row["text_ocr"],
                                    "t31", POOLS["t31"], return_raw=True)
        print(f"  id={vid}  PRED={pred}  (sin GT - test)")
        print(f"  RAW (truncated)={rraw[:300]!r}")
print("OK" if smoke_ids else "NO smoke samples")


--- Smoke test predict_video (T3.1) | BACKEND=gemma | RUN_MODE=test ---
    MAX_NEW_TOKENS = {'t31': 512, 't32': 512, 't33': 1024}
  id=320001  PRED=YES  (sin GT - test)
  RAW (truncated)='YES'
OK


In [12]:
# === 5.3 - Smoke test de presupuesto de tokens ==============================
# Cuenta los tokens reales del prompt completo (system + few-shot pool + query)
# despues de incluir los tokens de video. Avisa si supera el 80% del max
# context del modelo y aborta si supera el 100%.
def _count_prompt_tokens(task="t31"):
    """Construye el mensaje de smoke y cuenta input_ids tras prepararlo."""
    base_pool = df_test if RUN_MODE == "test" else df_train
    test_q = base_pool[~base_pool["id_EXIST"].isin(POOL_IDS)].iloc[0]
    msgs = build_messages(task, test_q["video_path"], test_q["text_ocr"], POOLS[task])
    inputs = _PREPARE_INPUTS(msgs)
    n_tok = inputs.input_ids.shape[1]
    return n_tok, test_q["id_EXIST"]

_max_ctx = (getattr(model.config, "max_position_embeddings", None)
            or getattr(processor.tokenizer, "model_max_length", None)
            or 32768)
# Algunos tokenizers reportan max_length=int(1e30); saturamos
if _max_ctx > 1_000_000:
    _max_ctx = 262_144   # contexto practico de Qwen3.5 / Gemma-4 (~256K)

print(f"--- Token budget smoke test (BACKEND={BACKEND}) ---")
print(f"  max_context (efectivo) = {_max_ctx:,} tokens")
for _t in ("t31", "t32", "t33"):
    try:
        _n, _vid = _count_prompt_tokens(_t)
    except Exception as _e:
        print(f"  [{_t}] no pudo medirse: {type(_e).__name__}: {_e}")
        continue
    _ratio = _n / _max_ctx
    _flag  = "OK"
    if _ratio > 1.0:    _flag = "OVERFLOW"
    elif _ratio > 0.8:  _flag = "WARN >80%"
    print(f"  [{_t}] {_n:>7,} tokens  ({100*_ratio:5.1f}% del max_ctx)  {_flag}  | smoke_id={_vid}")
    assert _ratio < 1.0, (
        f"Prompt T{_t} excede el contexto: {_n}/{_max_ctx}. "
        "Reduce VIDEO_FPS / VIDEO_MAX_PIXELS o el numero de shots del pool."
    )


--- Token budget smoke test (BACKEND=gemma) ---
  max_context (efectivo) = 262,144 tokens
  [t31]  18,728 tokens  (  7.1% del max_ctx)  OK  | smoke_id=320001
  [t32]  19,545 tokens  (  7.5% del max_ctx)  OK  | smoke_id=320001
  [t33]  36,328 tokens  ( 13.9% del max_ctx)  OK  | smoke_id=320001


## 6 - Loop de inferencia jerarquico con checkpointing

Mismo flujo que QWEN_IM_F: T3.1 sobre todos los videos -> T3.2/T3.3 solo sobre los YES de T3.1. Para los NO se asigna automaticamente `T3.2 = "NO"` y `T3.3 = ["NO"]`.

Ahorra ~50% de llamadas en T3.2/T3.3 (que son las mas caras: cada video se procesa de cero en cada llamada).

In [13]:
# === 6.0 - Helpers: checkpointing, sanity sample, get_eval_df, run_inference =
CKPT_EVERY = 50  # video es lento; guarda checkpoint mas a menudo

_CKPT_SUFFIX = "test" if RUN_MODE == "test" else f"sanity{MAX_EVAL_VIDS or 'all'}"
CKPT_PATHS = {t: CKPT_DIR / f"predictions_{t}_{_CKPT_SUFFIX}_partial.json"
              for t in ["t31", "t32", "t33"]}

def _load_ckpt(task):
    p = CKPT_PATHS[task]
    if not p.exists(): return {}
    return json.load(open(p, "r", encoding="utf-8"))

def _save_ckpt(task, preds):
    with open(CKPT_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False)

def _task_fallback(task):
    """Etiqueta a usar cuando el predict_video lanza excepcion."""
    if task == "t33": return [T33_FALLBACK_CATEGORY]
    if task == "t32": return "DIRECT"
    return "NO"

# === Sample sanity ESTRATIFICADO por t31_hard ================================
# Sustituye al .sample(n=MAX_EVAL_VIDS, random_state=SEED) que perdia
# representatividad de la clase YES / NO.
_SANITY_SAMPLE = {"ids": None}
def _get_sanity_sample_ids():
    if _SANITY_SAMPLE["ids"] is None:
        base = df_train[~df_train["id_EXIST"].isin(POOL_IDS) & df_train["vid_ok"]]
        base = base[base["t31_hard"].notna()]
        if MAX_EVAL_VIDS is not None and len(base) > MAX_EVAL_VIDS:
            try:
                from sklearn.model_selection import train_test_split
                strata = base["t31_hard"].astype(int)
                sub, _ = train_test_split(
                    base,
                    train_size=MAX_EVAL_VIDS,
                    stratify=strata,
                    random_state=SEED,
                )
                base = sub
            except (ValueError, ImportError) as _e:
                # fallback no estratificado si falla la estratificacion
                print(f"[warn] sample estratificado fallo ({type(_e).__name__}); "
                      f"usando .sample sin stratify")
                base = base.sample(n=MAX_EVAL_VIDS, random_state=SEED)
        _SANITY_SAMPLE["ids"] = set(base["id_EXIST"].astype(str))
    return _SANITY_SAMPLE["ids"]

def get_eval_df(task):
    if RUN_MODE == "test":
        sub = df_test[df_test["vid_ok"]].copy()
        if MAX_EVAL_VIDS is not None and len(sub) > MAX_EVAL_VIDS:
            sub = sub.sample(n=MAX_EVAL_VIDS, random_state=SEED)
        return sub.reset_index(drop=True)
    sample_ids = _get_sanity_sample_ids()
    base = df_train[df_train["id_EXIST"].isin(sample_ids) & df_train["vid_ok"]]
    if task == "t31":
        sub = base[base["t31_hard"].notna()]
    elif task == "t32":
        sub = base[base["t32_hard"].notna()]
    else:
        sub = base
    return sub.copy().reset_index(drop=True)

def run_inference(task, target_ids=None):
    """Loop de inferencia con checkpointing, VRAM cleanup y excepciones tipadas."""
    if target_ids is None:
        eval_df = get_eval_df(task)
    else:
        ts = {str(x) for x in target_ids}
        eval_df = df[df["id_EXIST"].astype(str).isin(ts) & df["vid_ok"]].copy().reset_index(drop=True)
    preds = _load_ckpt(task)
    todo = eval_df[~eval_df["id_EXIST"].isin(preds.keys())].copy()
    print()
    print(f"=== INFERENCIA {task.upper()} | stage={STAGE} | mode={RUN_MODE} ===")
    print(f"  a predecir   : {len(eval_df):,}")
    print(f"  ya predichos : {len(preds):,}")
    print(f"  pendientes   : {len(todo):,}")
    if len(todo) == 0:
        print("  Nada que hacer."); return preds
    pbar = tqdm(todo.itertuples(index=False), total=len(todo), desc=f"infer {task}")
    t0 = time.time(); n_done = 0
    for row in pbar:
        vid = row.id_EXIST
        try:
            pred = predict_video(vid, row.video_path, row.text_ocr, task, POOLS[task])
        except torch.cuda.OutOfMemoryError as e:
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
            print(); print(f"[OOM] {vid}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<OOM: {e}>", str(pred), exc_type="OutOfMemoryError")
        except FileNotFoundError as e:
            print(); print(f"[FileNotFound] {vid}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<FileNotFoundError: {e}>", str(pred),
                          exc_type="FileNotFoundError")
        except RuntimeError as e:
            print(); print(f"[RuntimeError] {vid}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<RuntimeError: {e}>", str(pred),
                          exc_type="RuntimeError")
        except Exception as e:
            print(); print(f"[err] {vid}: {type(e).__name__}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<{type(e).__name__}: {e}>", str(pred),
                          exc_type=type(e).__name__)
        preds[vid] = pred
        n_done += 1
        if n_done % CKPT_EVERY == 0:
            _save_ckpt(task, preds)
            # VRAM cleanup periodico
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
            elapsed = time.time() - t0
            rate = n_done / max(elapsed, 1e-6)
            pbar.set_postfix({"vid/s": f"{rate:.2f}",
                              "fbacks": f"{_fallback_count[task]}/{_total_count[task]}"})
    _save_ckpt(task, preds)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    print()
    print(f"  fallbacks {task}: {_fallback_count[task]} / {_total_count[task]}")
    return preds


In [14]:
# === 6.1 - Baseline majority (antes del loop jerarquico) ====================
# Genera tres ficheros pred_t3X_hard_majority_<suffix>.json con un baseline
# trivial. Sirve como floor de comparacion: si tu modelo no supera majority,
# algo va mal en los prompts o en el dataset.
#
#   T3.1: clase mayoritaria del split TRAIN (calculada in situ).
#   T3.2: DIRECT siempre (solo se aplica donde T3.1 majority predice YES;
#         si majority de T3.1 es NO -> T3.2 = NO).
#   T3.3: T33_FALLBACK_CATEGORY siempre (idem condicional sobre T3.1).
T31_MAJORITY = (df_train["t31_hard"].dropna().astype(int).map(T31_INT_TO_STR)
                  .value_counts().idxmax())
T32_MAJORITY = "DIRECT"
T33_MAJORITY = ["STEREOTYPING-DOMINANCE"]   # constante por requisito del review
print(f"T3.1 majority en TRAIN  = {T31_MAJORITY}")
print(f"T3.2 majority (constante) = {T32_MAJORITY}")
print(f"T3.3 majority (constante) = {T33_MAJORITY}")

def _build_majority_preds():
    eval_t31 = get_eval_df("t31")
    eval_t32 = get_eval_df("t32")
    eval_t33 = get_eval_df("t33")
    union_ids = (set(eval_t31["id_EXIST"]) | set(eval_t32["id_EXIST"])
                 | set(eval_t33["id_EXIST"]))
    preds31 = {vid: T31_MAJORITY for vid in union_ids}
    if T31_MAJORITY == "NO":
        preds32 = {vid: "NO" for vid in eval_t32["id_EXIST"]}
        preds33 = {vid: ["NO"] for vid in eval_t33["id_EXIST"]}
    else:
        preds32 = {vid: T32_MAJORITY for vid in eval_t32["id_EXIST"]}
        preds33 = {vid: T33_MAJORITY for vid in eval_t33["id_EXIST"]}
    return preds31, preds32, preds33

majority_preds_t31, majority_preds_t32, majority_preds_t33 = _build_majority_preds()
MAJ_PRED_PATHS = {
    "t31": PRED_DIR / f"pred_t31_hard_majority_{_CKPT_SUFFIX}.json",
    "t32": PRED_DIR / f"pred_t32_hard_majority_{_CKPT_SUFFIX}.json",
    "t33": PRED_DIR / f"pred_t33_hard_majority_{_CKPT_SUFFIX}.json",
}

def _build_pyevall_majority(preds, task):
    return [{"test_case": "EXIST2025", "id": str(vid), "value": v}
            for vid, v in preds.items()]

for task, preds in [("t31", majority_preds_t31),
                    ("t32", majority_preds_t32),
                    ("t33", majority_preds_t33)]:
    pj = _build_pyevall_majority(preds, task)
    with open(MAJ_PRED_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(pj, f, ensure_ascii=False, indent=2)
    print(f"[majority {task}] {len(pj):,} predicciones -> {MAJ_PRED_PATHS[task].name}")


T3.1 majority en TRAIN  = NO
T3.2 majority (constante) = DIRECT
T3.3 majority (constante) = ['STEREOTYPING-DOMINANCE']
[majority t31] 674 predicciones -> pred_t31_hard_majority_test.json
[majority t32] 674 predicciones -> pred_t32_hard_majority_test.json
[majority t33] 674 predicciones -> pred_t33_hard_majority_test.json


In [15]:
# === 6.2 - Loop de inferencia JERARQUICO (T3.1 -> T3.2/T3.3 sobre YES) ======
eval_t31_df = get_eval_df("t31")
eval_t32_df = get_eval_df("t32")
eval_t33_df = get_eval_df("t33")
union_ids = (set(eval_t31_df["id_EXIST"]) | set(eval_t32_df["id_EXIST"])
             | set(eval_t33_df["id_EXIST"]))
print()
print(f"=== UNION eval set: {len(union_ids):,} videos (T3.1 corre sobre TODOS) ===")
print(f"   T3.1 eval: {len(eval_t31_df):,}  T3.2 eval: {len(eval_t32_df):,}  T3.3 eval: {len(eval_t33_df):,}")

# 1) T3.1 sobre la union
preds_t31 = run_inference("t31", target_ids=union_ids)

# 2) Sexistas
sexist_ids     = {vid for vid, v in preds_t31.items() if v == "YES"}
non_sexist_ids = {vid for vid, v in preds_t31.items() if v == "NO"}
print()
print(f"=== T3.1 -> {len(sexist_ids):,} YES ({100*len(sexist_ids)/max(1,len(preds_t31)):.1f}%) | "
      f"{len(non_sexist_ids):,} NO ===")

# 3) T3.2 solo sobre sexistas
t32_targets = sexist_ids & set(eval_t32_df["id_EXIST"])
print(f"   T3.2 modelo sobre {len(t32_targets):,} videos")
preds_t32 = run_inference("t32", target_ids=t32_targets)
for vid in eval_t32_df["id_EXIST"]:
    if preds_t31.get(vid) == "NO":
        preds_t32[vid] = "NO"
    elif vid not in preds_t32:
        row = df.loc[df["id_EXIST"] == vid].iloc[0]
        preds_t32[vid] = predict_video(vid, row["video_path"], row["text_ocr"],
                                         "t32", POOLS["t32"])
_save_ckpt("t32", preds_t32)

# 4) T3.3 idem
t33_targets = sexist_ids & set(eval_t33_df["id_EXIST"])
print()
print(f"   T3.3 modelo sobre {len(t33_targets):,} videos")
preds_t33 = run_inference("t33", target_ids=t33_targets)
for vid in eval_t33_df["id_EXIST"]:
    if preds_t31.get(vid) == "NO":
        preds_t33[vid] = ["NO"]
    elif vid not in preds_t33:
        row = df.loc[df["id_EXIST"] == vid].iloc[0]
        preds_t33[vid] = predict_video(vid, row["video_path"], row["text_ocr"],
                                         "t33", POOLS["t33"])
_save_ckpt("t33", preds_t33)

print()
print("=== Resumen jerarquia ===")
print(f"  preds_t31 final: {len(preds_t31):,}")
print(f"  preds_t32 final: {len(preds_t32):,}")
print(f"  preds_t33 final: {len(preds_t33):,}")



=== UNION eval set: 674 videos (T3.1 corre sobre TODOS) ===
   T3.1 eval: 674  T3.2 eval: 674  T3.3 eval: 674

=== INFERENCIA T31 | stage=submission | mode=test ===
  a predecir   : 674
  ya predichos : 0
  pendientes   : 674


infer t31:   0%|          | 0/674 [00:00<?, ?it/s]


[err] 320057: ValueError: Video can't be sampled. The `num_frames=32` exceeds `total_num_frames=15`. 

[err] 320090: ValueError: Video can't be sampled. The `num_frames=32` exceeds `total_num_frames=10`. 

[err] 320235: ValueError: Video can't be sampled. The `num_frames=32` exceeds `total_num_frames=15`. 

[RuntimeError] 420007: Failed to read frame from input file: Invalid data found when processing input

[err] 420047: ValueError: Video can't be sampled. The `num_frames=32` exceeds `total_num_frames=15`. 

[RuntimeError] 420153: Failed to read frame from input file: Invalid data found when processing input

[RuntimeError] 420283: Failed to read frame from input file: Invalid data found when processing input

[RuntimeError] 420368: Failed to read frame from input file: Invalid data found when processing input

  fallbacks t31: 0 / 667

=== T3.1 -> 308 YES (45.7%) | 366 NO ===
   T3.2 modelo sobre 308 videos

=== INFERENCIA T32 | stage=submission | mode=test ===
  a predecir   : 308


infer t32:   0%|          | 0/308 [00:00<?, ?it/s]


  fallbacks t32: 0 / 308

   T3.3 modelo sobre 308 videos

=== INFERENCIA T33 | stage=submission | mode=test ===
  a predecir   : 308
  ya predichos : 0
  pendientes   : 308


infer t33:   0%|          | 0/308 [00:00<?, ?it/s]


[OOM] 320001: CUDA out of memory. Tried to allocate 39.99 GiB. GPU 0 has a total capacity of 94.97 GiB of which 33.62 GiB is free. Including non-PyTorch memory, this process has 61.34 GiB memory in use. Of the allocated memory 60.69 GiB is allocated by PyTorch, and 8.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

[OOM] 320002: CUDA out of memory. Tried to allocate 39.99 GiB. GPU 0 has a total capacity of 94.97 GiB of which 33.43 GiB is free. Including non-PyTorch memory, this process has 61.54 GiB memory in use. Of the allocated memory 60.69 GiB is allocated by PyTorch, and 210.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See

### 6.3 - Retry T3.3 tras OOM (recovery)

Si la inferencia de T3.3 (cell 6.2) se quedó OOM a mitad, ejecuta esta celda
para reanudar **solo T3.3** sin tener que reejecutar T3.1 ni T3.2:

1. Libera VRAM (gc + `torch.cuda.empty_cache`).
2. Restaura `preds_t31` y `preds_t32` desde los checkpoints en disco.
3. Recalcula `sexist_ids` y `t33_targets`.
4. Lanza `run_inference("t33", ...)` — gracias al checkpointing, **solo procesa
   los vídeos pendientes** (los ya predichos en el partial `predictions_t33_*_partial.json` se preservan).
5. Aplica la jerarquía (T3.1 = NO → T3.3 = `["NO"]`).
6. Si un vídeo individual OOMea, se aplica `_task_fallback("t33")` y se continúa.


In [21]:
# === 6.3 - Retry T3.3 tras OOM (reanuda desde checkpoint) ===================
import gc, torch

# 1) VRAM cleanup agresivo
print("--- VRAM cleanup ---")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    _vram = torch.cuda.memory_allocated() / 1e9
    _vram_r = torch.cuda.memory_reserved()  / 1e9
    print(f"  VRAM alocada: {_vram:.2f} GB | reservada: {_vram_r:.2f} GB")

# 2) Restaurar preds_t31 y preds_t32 desde disk si el kernel se reinicio
if "preds_t31" not in globals() or not preds_t31:
    preds_t31 = _load_ckpt("t31")
    print(f"  preds_t31 cargado desde disk: {len(preds_t31)} entries")
else:
    print(f"  preds_t31 ya en memoria: {len(preds_t31)} entries")
if "preds_t32" not in globals() or not preds_t32:
    preds_t32 = _load_ckpt("t32")
    print(f"  preds_t32 cargado desde disk: {len(preds_t32)} entries")
else:
    print(f"  preds_t32 ya en memoria: {len(preds_t32)} entries")

# 3) Recalcular sexist_ids desde preds_t31
sexist_ids = {vid for vid, v in preds_t31.items() if v == "YES"}
print(f"  sexist_ids (T3.1 = YES): {len(sexist_ids)}")

# 4) Eval set T3.3 + targets
eval_t33_df = get_eval_df("t33")
t33_targets = sexist_ids & set(eval_t33_df["id_EXIST"])
print(f"  T3.3 targets (interseccion sexistas x eval_t33): {len(t33_targets)}")

# 5) Estado del partial cacheado
_partial_t33 = _load_ckpt("t33")
_pending = t33_targets - set(_partial_t33.keys())
print(f"  T3.3 partial cacheado: {len(_partial_t33)} entries")
print(f"  T3.3 pendientes:      {len(_pending)}  <- los que faltaban tras el OOM")
if not _pending:
    print("  Nada pendiente. preds_t33 esta completo — salta a la celda 7.1")

# 6) Resume inferencia T3.3 (run_inference reaprovecha el partial automaticamente)
preds_t33 = run_inference("t33", target_ids=t33_targets)

# 7) Aplicar jerarquia: T3.1=NO -> T3.3=["NO"]. Para casos raros (eval_t33 sin
#    pred_t31 valida), intentamos predecir con try/except y fallback en OOM.
for vid in eval_t33_df["id_EXIST"]:
    if preds_t31.get(vid) == "NO":
        preds_t33[vid] = ["NO"]
    elif vid not in preds_t33:
        row = df.loc[df["id_EXIST"] == vid].iloc[0]
        try:
            preds_t33[vid] = predict_video(vid, row["video_path"], row["text_ocr"],
                                             "t33", POOLS["t33"])
        except (torch.cuda.OutOfMemoryError, RuntimeError) as _e:
            print(f"  [OOM/RuntimeError] {vid}: fallback _task_fallback('t33')")
            preds_t33[vid] = _task_fallback("t33")
            torch.cuda.empty_cache()
_save_ckpt("t33", preds_t33)

# 8) Resumen final
print()
print("=== T3.3 final ===")
print(f"  total preds_t33   : {len(preds_t33):,}")
_n_no   = sum(1 for v in preds_t33.values() if v == ["NO"])
_n_sex  = sum(1 for v in preds_t33.values() if v != ["NO"])
_n_fail = sum(1 for v in preds_t33.values() if v == [T33_FALLBACK_CATEGORY])
print(f"  con ['NO']        : {_n_no:,}")
print(f"  con cats sexistas : {_n_sex:,}")
print(f"  con fallback      : {_n_fail:,}  (errores OOM individuales)")
print()
print(">> Ahora puedes continuar con la celda 7.1 (PyEvALL hard JSON builder)")


--- VRAM cleanup ---
  VRAM alocada: 62.55 GB | reservada: 62.57 GB
  preds_t31 ya en memoria: 674 entries
  preds_t32 ya en memoria: 674 entries
  sexist_ids (T3.1 = YES): 308
  T3.3 targets (interseccion sexistas x eval_t33): 308
  T3.3 partial cacheado: 0 entries
  T3.3 pendientes:      308  <- los que faltaban tras el OOM

=== INFERENCIA T33 | stage=submission | mode=test ===
  a predecir   : 308
  ya predichos : 0
  pendientes   : 308


infer t33:   0%|          | 0/308 [00:00<?, ?it/s]


  fallbacks t33: 0 / 308

=== T3.3 final ===
  total preds_t33   : 674
  con ['NO']        : 366
  con cats sexistas : 308
  con fallback      : 140  (errores OOM individuales)

>> Ahora puedes continuar con la celda 7.1 (PyEvALL hard JSON builder)


In [20]:
# === Fix OOM T3.3: reducir pool + cleanup + env var ========================
import os, gc, torch

# 1) Habilitar expandable_segments (mitiga fragmentación si hay)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))

# 2) Cleanup VRAM agresivo
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"VRAM: alocada={torch.cuda.memory_allocated()/1e9:.2f} GB | "
          f"reservada={torch.cuda.memory_reserved()/1e9:.2f} GB")

# 3) Reducir pool T3.3 a 6 ejemplos (de 10 originales) — guardamos backup
_POOL_T33_FULL = list(POOLS["t33"])    # backup en RAM por si quieres restaurar
print(f"\nPool T3.3 original: {len(_POOL_T33_FULL)} ejemplos")
for ex in _POOL_T33_FULL:
    lbl = ex.get("label") or ex.get("labels")
    print(f"    id={ex['id']:>10}  cons={ex.get('consensus', 0):.2f}  label={lbl}")

# Reducimos: 1 ejemplo por cada una de las 5 categorías sexistas (5 totales)
# preservando los de mayor consenso. Si el pool original ya tiene una sola
# representación por categoría, esto NO debería degradar mucho la calidad.
_seen_cats = set()
_pool_reduced = []
for ex in sorted(_POOL_T33_FULL, key=lambda e: -e.get("consensus", 0)):
    labs = ex.get("labels") or []
    main_cat = labs[0] if labs else None
    if main_cat and main_cat not in _seen_cats:
        _seen_cats.add(main_cat)
        _pool_reduced.append(ex)
    if len(_pool_reduced) >= 5:
        break

POOLS["t33"] = _pool_reduced
print(f"\nPool T3.3 reducido: {len(POOLS['t33'])} ejemplos (1 por cat sexista, mayor consenso)")
for ex in POOLS["t33"]:
    lbl = ex.get("label") or ex.get("labels")
    print(f"    id={ex['id']:>10}  cons={ex.get('consensus', 0):.2f}  label={lbl}")

# 4) Reducir MAX_NEW_TOKENS T3.3 (limita KV cache growth durante generation)
MAX_NEW_TOKENS["t33"] = 384   # bajo de 1024 → suficiente para list de 1-3 cats
print(f"\nMAX_NEW_TOKENS['t33'] = {MAX_NEW_TOKENS['t33']}  (era 1024)")

# 5) Borrar el partial T3.3 corrupto + reset contadores
if CKPT_PATHS["t33"].exists():
    CKPT_PATHS["t33"].unlink()
    print(f"\nBorrado partial t33: {CKPT_PATHS['t33'].name}")
_fallback_count["t33"] = 0
_total_count["t33"]    = 0
if "preds_t33" in globals():
    del preds_t33

print("\n>> Ahora ejecuta la celda 6.3 (retry). Cada video usará ~40% menos VRAM.")
print(">> Si vuelve a OOMear: reduce más con `POOLS['t33'] = POOLS['t33'][:3]`")


PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True
VRAM: alocada=62.55 GB | reservada=62.57 GB

Pool T3.3 original: 12 ejemplos
    id=    120065  cons=1.00  label=['IDEOLOGICAL-INEQUALITY']
    id=    120066  cons=1.00  label=['IDEOLOGICAL-INEQUALITY']
    id=    120005  cons=1.00  label=['STEREOTYPING-DOMINANCE']
    id=    120009  cons=1.00  label=['STEREOTYPING-DOMINANCE']
    id=    120074  cons=1.00  label=['OBJECTIFICATION']
    id=    120263  cons=1.00  label=['OBJECTIFICATION']
    id=    120080  cons=1.00  label=['SEXUAL-VIOLENCE']
    id=    120238  cons=1.00  label=['SEXUAL-VIOLENCE']
    id=    120002  cons=1.00  label=['MISOGYNY-NON-SEXUAL-VIOLENCE']
    id=    120018  cons=1.00  label=['MISOGYNY-NON-SEXUAL-VIOLENCE']
    id=    220437  cons=1.00  label=['IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 'SEXUAL-VIOLENCE']
    id=    220764  cons=1.00  label=['IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 'OBJECTIFICATION']

Pool T3.3 reducido: 5 ejemplos (1 por cat

## 7 - Generacion de los JSON PyEvALL HARD

Tres archivos `pred_t3X_hard_<stage>.json` con `test_case = "EXIST2025"` y sanity check de invariantes (mismo formato que la submission de memes pero con `task3_X` en el filename).

In [22]:
# === 7.1 - PyEvALL hard JSON builder + sanity check =========================
PYEVALL_TEST_CASE = "EXIST2025"   # OBLIGATORIO segun guidelines pag. 12

def build_pyevall_hard(preds_dict, task):
    out = []
    for vid, pred in preds_dict.items():
        out.append({"test_case": PYEVALL_TEST_CASE, "id": str(vid), "value": pred})
    return out

def _sanity_check_pred(pred_list, task):
    valid_t32 = {"NO", "DIRECT", "JUDGEMENTAL"}
    valid_t33 = set(SEXISM_CATS) | {"NO"}
    for p in pred_list:
        assert "id" in p, "missing 'id'"
        assert p.get("test_case") == "EXIST2025", f"wrong test_case: {p.get('test_case')}"
        v = p["value"]
        if task == "t31":
            assert v in {"NO", "YES"}, f"T3.1 invalid: {v}"
        elif task == "t32":
            assert v in valid_t32, f"T3.2 invalid: {v}"
        elif task == "t33":
            assert isinstance(v, list), f"T3.3 not list: {v}"
            for x in v:
                assert x != "UNKNOWN", "UNKNOWN predicted"
                assert x in valid_t33, f"T3.3 invalid cat: {x}"
            if "NO" in v:
                assert v == ["NO"], f"T3.3 mixes NO with cats: {v}"

PRED_PATHS = {
    "t31": PRED_DIR / f"pred_t31_hard_{_CKPT_SUFFIX}.json",
    "t32": PRED_DIR / f"pred_t32_hard_{_CKPT_SUFFIX}.json",
    "t33": PRED_DIR / f"pred_t33_hard_{_CKPT_SUFFIX}.json",
}
for task, preds in [("t31", preds_t31), ("t32", preds_t32), ("t33", preds_t33)]:
    pj = build_pyevall_hard(preds, task)
    _sanity_check_pred(pj, task)
    with open(PRED_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(pj, f, ensure_ascii=False, indent=2)
    print(f"[{task}] {len(pj):,} predicciones -> {PRED_PATHS[task].name}")

if RUN_MODE == "test":
    expected_ids = set(df_test["id_EXIST"].astype(str))
    for task, preds in [("t31", preds_t31), ("t32", preds_t32), ("t33", preds_t33)]:
        got = set(preds.keys())
        miss = expected_ids - got
        assert not miss, f"[{task}] faltan {len(miss)} predicciones (e.g. {list(miss)[:3]})"
    print(f"\nOK Cobertura test: {len(expected_ids):,} videos en cada subtask.")

[t31] 674 predicciones -> pred_t31_hard_test.json
[t32] 674 predicciones -> pred_t32_hard_test.json
[t33] 674 predicciones -> pred_t33_hard_test.json

OK Cobertura test: 674 videos en cada subtask.


## 8 - Evaluacion + comparativa detallada

En `train_eval`: PyEvALL ICM/ICMNorm/FMeasure + matriz de confusion + classification report. En `test`: solo distribuciones (no hay gold).

In [24]:
# === 8.1 - Instalacion + import de PyEvALL ==================================
try:
    from pyevall.evaluation import PyEvALLEvaluation
    from pyevall.utils.utils import PyEvALLUtils
except ImportError:
    import subprocess as _sp, sys as _sys, importlib as _il
    print("Instalando PyEvALL...")
    try:
        _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "PyEvALL"])
    except Exception:
        _sp.check_call([_sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/UNEDLENAR/PyEvALL.git"])
    _il.invalidate_caches()
    from pyevall.evaluation import PyEvALLEvaluation
    from pyevall.utils.utils import PyEvALLUtils
print("PyEvALL listo:", PyEvALLEvaluation.__module__)

Instalando PyEvALL...
PyEvALL listo: pyevall.evaluation


In [25]:
# === 8.2 - Helpers de evaluacion hard-hard ==================================
HIERARCHIES_HARD = {
    "t31": None,
    "t32": {"YES": ["DIRECT", "JUDGEMENTAL"], "NO": []},
    "t33": {"YES": list(SEXISM_CATS), "NO": []},
}
METRICS_HARD = {
    "t31": ["ICM", "ICMNorm", "FMeasure"],
    "t32": ["ICM", "ICMNorm", "FMeasure"],
    "t33": ["ICM", "ICMNorm", "FMeasure"],
}

def build_pyevall_gold_hard(ids, task):
    out = []
    for vid in ids:
        m = raw[str(vid)]
        if task == "t31":
            v_int = hard_t31(m.get("labels_task3_1") or [])
            if v_int is None: continue
            v = T31_INT_TO_STR[v_int]
        elif task == "t32":
            v_int = hard_t32(m.get("labels_task3_2") or [])
            if v_int is None: continue
            v = T32_INT_TO_STR[v_int]
        else:
            multi = hard_t33_multihot(m.get("labels_task3_3") or [])
            picks = [SEXISM_CATS[k] for k in range(5) if multi[k] > 0.5]
            v = picks if picks else ["NO"]
        out.append({"test_case": PYEVALL_TEST_CASE, "id": str(vid), "value": v})
    return out

def _walk_for_metric(d, metric_name, depth=0, max_depth=8):
    if depth > max_depth: return None
    if isinstance(d, dict):
        if metric_name in d:
            v = d[metric_name]
            if isinstance(v, (int, float)): return float(v)
            if isinstance(v, dict):
                nums = []
                for vv in v.values():
                    if isinstance(vv, (int, float)): nums.append(vv)
                    elif isinstance(vv, dict):
                        for vvv in vv.values():
                            if isinstance(vvv, (int, float)): nums.append(vvv)
                if nums: return sum(nums)/len(nums)
        for k, v in d.items():
            r = _walk_for_metric(v, metric_name, depth+1, max_depth)
            if r is not None: return r
    elif isinstance(d, list):
        nums = []
        for it in d:
            r = _walk_for_metric(it, metric_name, depth+1, max_depth)
            if r is not None: nums.append(r)
        if nums: return sum(nums)/len(nums)
    return None

def _parse_pyevall_report_hard(report, metric_names=("ICM","ICMNorm","FMeasure")):
    if report is None: return {}
    out = {}
    for attr in ("report","dict_report","json_report","metrics_data",
                  "_report","data","result","results"):
        ra = getattr(report, attr, None)
        if ra is None: continue
        if hasattr(ra, "columns"):
            for m in metric_names:
                if m in ra.columns and m not in out:
                    vals = ra[m].dropna()
                    if len(vals): out[m] = float(vals.mean())
        else:
            for m in metric_names:
                if m in out: continue
                v = _walk_for_metric(ra, m)
                if v is not None: out[m] = v
    if all(m in out for m in metric_names): return out
    import io as _io, contextlib as _ctx
    buf = _io.StringIO()
    with _ctx.redirect_stdout(buf):
        for fn in ("print_report_tsv","print_report"):
            try: getattr(report, fn)(); break
            except Exception: continue
    text = buf.getvalue()
    aliases = {"ICM":["ICM"], "ICMNorm":["ICMNorm","ICM-Norm","ICM Norm"],
                "FMeasure":["FMeasure","F-Measure","F1"]}
    for m in metric_names:
        if m in out: continue
        for a in aliases.get(m, [m]):
            esc = re.escape(a)
            mt = re.search(rf"\b{esc}\b\s*[:=]\s*(-?\d+\.\d+)", text) \
                or re.search(rf"\b{esc}\b[^\d-]{{0,100}}(-?\d+\.\d+)", text)
            if mt:
                try: out[m] = float(mt.group(1)); break
                except ValueError: pass
    return out

def evaluate_pyevall_hard(pred_list, gold_list, task):
    gold_ids = {g["id"] for g in gold_list}
    pred_filt = [p for p in pred_list if p["id"] in gold_ids]
    pred_path = PYE_WORK / f"_tmp_pred_{task}_hard.json"
    gold_path = PYE_WORK / f"_tmp_gold_{task}_hard.json"
    with open(pred_path, "w", encoding="utf-8") as f: json.dump(pred_filt, f)
    with open(gold_path, "w", encoding="utf-8") as f: json.dump(gold_list, f)
    params = {PyEvALLUtils.PARAM_REPORT: PyEvALLUtils.PARAM_OPTION_REPORT_DATAFRAME}
    h = HIERARCHIES_HARD[task]
    if h is not None: params[PyEvALLUtils.PARAM_HIERARCHY] = h
    test = PyEvALLEvaluation()
    rep = test.evaluate(str(pred_path), str(gold_path), METRICS_HARD[task], **params)
    return _parse_pyevall_report_hard(rep, tuple(METRICS_HARD[task]))

In [26]:
# === 8.3 - Evaluacion + reporte (modelo principal + baseline majority) =====
if RUN_MODE == "train_eval":
    summary_rows = []

    def _eval_one_run(label, pred_paths, model_id):
        """Evalua una run (sea modelo principal o majority) con PyEvALL."""
        results = {}
        print()
        print(f"--- Evaluando run: {label}  (model={model_id}) ---")
        for task in ["t31","t32","t33"]:
            pred_list = json.load(open(pred_paths[task], "r", encoding="utf-8"))
            pred_ids  = [p["id"] for p in pred_list]
            gold_list = build_pyevall_gold_hard(pred_ids, task)
            metrics   = evaluate_pyevall_hard(pred_list, gold_list, task)
            results[task] = metrics
            print(f"[{task.upper()}]  n_pred={len(pred_list):,}  n_gold={len(gold_list):,}")
            for m in METRICS_HARD[task]:
                v = metrics.get(m)
                print(f"    {m:>10}: {v:.4f}" if v is not None else f"    {m:>10}: <missing>")
        for task in ["t31","t32","t33"]:
            r = results[task]
            summary_rows.append({"run": label, "task": task,
                                 "ICM": r.get("ICM"),
                                 "ICMNorm": r.get("ICMNorm"),
                                 "FMeasure": r.get("FMeasure"),
                                 "model": model_id})
        return results

    # 1) Modelo principal
    results = _eval_one_run("model", PRED_PATHS, MODEL_ID)

    # 2) Baseline majority
    try:
        results_majority = _eval_one_run("majority", MAJ_PRED_PATHS, "majority")
    except Exception as _e:
        print(f"[warn] no se pudo evaluar majority: {type(_e).__name__}: {_e}")

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(METR_DIR / "metrics_summary.csv", index=False)
    print()
    print(f"=== RESUMEN COMPARATIVO (modelo vs majority) ===")
    print(summary_df.to_string(index=False))
    print(f"\nGuardado en {METR_DIR / 'metrics_summary.csv'}")
else:
    print(f"=== RUN_MODE=test: PyEvALL eval omitido (test set sin labels) ===")
    print(f"Modelo: {MODEL_ID}\n")
    for task, preds in [("t31", preds_t31), ("t32", preds_t32), ("t33", preds_t33)]:
        if task == "t33":
            cnt = Counter()
            for v in preds.values():
                key = "NO" if v == ["NO"] else "+".join(sorted(v))
                cnt[key] += 1
        else:
            cnt = Counter(preds.values())
        print(f"[{task.upper()}]  n={len(preds):,}")
        total = sum(cnt.values())
        for k, v in cnt.most_common():
            print(f"    {str(k):<60}  {v:>5}  ({100*v/total:5.1f}%)")
        print()


2026-05-04 15:26:44,573 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-05-04 15:26:44,588 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-04 15:26:44,606 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-05-04 15:26:44,607 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-04 15:26:44,624 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-05-04 15:26:44,642 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
cargado 29

[T31]  n_pred=200  n_gold=200
           ICM: 0.1305
       ICMNorm: 0.5653
      FMeasure: 0.7025
2026-05-04 15:26:44,674 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-05

In [27]:
# === 8.3b - Comparativa detallada predicciones vs gold (solo train_eval) ====
if RUN_MODE != "train_eval":
    print("Comparativa detallada solo disponible en STAGE='sanity'.")
else:
    from sklearn.metrics import (confusion_matrix, classification_report,
                                  precision_recall_fscore_support, f1_score)
    from sklearn.preprocessing import MultiLabelBinarizer

    def _compare_t31():
        rows = []
        for vid, pred in preds_t31.items():
            gt_int = df.loc[df["id_EXIST"]==vid, "t31_hard"].iloc[0]
            if pd.isna(gt_int): continue
            gt = T31_INT_TO_STR[int(gt_int)]
            rows.append({"id": vid, "gt": gt, "pred": pred, "ok": pred == gt})
        cmp_df = pd.DataFrame(rows)
        labels = ["NO","YES"]
        cm = confusion_matrix(cmp_df["gt"], cmp_df["pred"], labels=labels)
        print()
        print("=== T3.1 (binary YES/NO) ===")
        print(f"  n_eval = {len(cmp_df):,}   accuracy = {cmp_df['ok'].mean():.4f}")
        print()
        print("Confusion matrix (filas=GT, cols=PRED):")
        print(pd.DataFrame(cm, index=[f"GT_{l}" for l in labels],
                                 columns=[f"PR_{l}" for l in labels]))
        print()
        print("Classification report:")
        print(classification_report(cmp_df["gt"], cmp_df["pred"],
                                     labels=labels, digits=4, zero_division=0))
        return cmp_df

    def _compare_t32():
        rows = []
        for vid, pred in preds_t32.items():
            gt_int = df.loc[df["id_EXIST"]==vid, "t32_hard"].iloc[0]
            if pd.isna(gt_int): continue
            gt = T32_INT_TO_STR[int(gt_int)]
            rows.append({"id": vid, "gt": gt, "pred": pred, "ok": pred == gt})
        cmp_df = pd.DataFrame(rows)
        labels = ["NO","DIRECT","JUDGEMENTAL"]
        cm = confusion_matrix(cmp_df["gt"], cmp_df["pred"], labels=labels)
        print()
        print("=== T3.2 (NO/DIRECT/JUDGEMENTAL) ===")
        print(f"  n_eval = {len(cmp_df):,}   accuracy = {cmp_df['ok'].mean():.4f}")
        print()
        print("Confusion matrix:")
        print(pd.DataFrame(cm, index=[f"GT_{l}" for l in labels],
                                 columns=[f"PR_{l}" for l in labels]))
        print()
        print("Classification report:")
        print(classification_report(cmp_df["gt"], cmp_df["pred"],
                                     labels=labels, digits=4, zero_division=0))
        return cmp_df

    def _compare_t33():
        all_classes = ["NO"] + SEXISM_CATS
        mlb = MultiLabelBinarizer(classes=all_classes)
        y_true_list, y_pred_list, ids = [], [], []
        for vid, pred in preds_t33.items():
            multi = df.loc[df["id_EXIST"]==vid, "t33_hard"].iloc[0]
            if multi is None: continue
            picks = [SEXISM_CATS[k] for k in range(5) if multi[k] > 0.5]
            gt = picks if picks else ["NO"]
            y_true_list.append(gt); y_pred_list.append(pred); ids.append(vid)
        y_true = mlb.fit_transform(y_true_list)
        y_pred = mlb.transform(y_pred_list)
        print()
        print("=== T3.3 (multi-label) ===")
        print(f"  n_eval = {len(ids):,}")
        print(f"  {'class':<32}  {'P':>7} {'R':>7} {'F1':>7} {'support':>8}")
        for i, cls in enumerate(all_classes):
            p, r, f, _ = precision_recall_fscore_support(
                y_true[:, i], y_pred[:, i], average="binary", zero_division=0)
            sup = int(y_true[:, i].sum())
            print(f"  {cls:<32}  {p:7.4f} {r:7.4f} {f:7.4f} {sup:>8d}")
        f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
        f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
        exact = sum(1 for yt, yp in zip(y_true_list, y_pred_list) if set(yt) == set(yp))
        print()
        print(f"  F1-macro    = {f1_macro:.4f}")
        print(f"  F1-micro    = {f1_micro:.4f}")
        print(f"  Exact match = {exact/len(ids):.4f}")
        return pd.DataFrame({"id": ids, "gt": y_true_list, "pred": y_pred_list})

    cmp_t31 = _compare_t31()
    cmp_t32 = _compare_t32()
    cmp_t33 = _compare_t33()

    err_dir = METR_DIR / "errors"; err_dir.mkdir(parents=True, exist_ok=True)
    for task, cdf in [("t31", cmp_t31), ("t32", cmp_t32)]:
        errs = cdf[cdf["pred"] != cdf["gt"]].copy()
        errs = errs.merge(df[["id_EXIST","lang","text_ocr"]],
                           left_on="id", right_on="id_EXIST", how="left")
        errs = errs[["id","lang","gt","pred","text_ocr"]]
        out = err_dir / f"errors_{task}.csv"
        errs.to_csv(out, index=False, encoding="utf-8")
        print()
        print(f"  Errores {task}: {len(errs)} guardados en {out}")
    err33 = cmp_t33[cmp_t33.apply(lambda r: set(r["gt"]) != set(r["pred"]), axis=1)].copy()
    err33 = err33.merge(df[["id_EXIST","lang","text_ocr"]],
                         left_on="id", right_on="id_EXIST", how="left")
    err33 = err33[["id","lang","gt","pred","text_ocr"]]
    out33 = err_dir / "errors_t33.csv"
    err33.to_csv(out33, index=False, encoding="utf-8")
    print(f"  Errores t33: {len(err33)} guardados en {out33}")


=== T3.1 (binary YES/NO) ===
  n_eval = 200   accuracy = 0.7150

Confusion matrix (filas=GT, cols=PRED):
        PR_NO  PR_YES
GT_NO      92      13
GT_YES     44      51

Classification report:
              precision    recall  f1-score   support

          NO     0.6765    0.8762    0.7635       105
         YES     0.7969    0.5368    0.6415        95

    accuracy                         0.7150       200
   macro avg     0.7367    0.7065    0.7025       200
weighted avg     0.7337    0.7150    0.7055       200


=== T3.2 (NO/DIRECT/JUDGEMENTAL) ===
  n_eval = 196   accuracy = 0.6684

Confusion matrix:
                PR_NO  PR_DIRECT  PR_JUDGEMENTAL
GT_NO              92         13               0
GT_DIRECT          28         33               2
GT_JUDGEMENTAL     14          8               6

Classification report:
              precision    recall  f1-score   support

          NO     0.6866    0.8762    0.7699       105
      DIRECT     0.6111    0.5238    0.5641        63
 J

In [ ]:
# === 8.4 - Reporte de fallbacks (calidad de prompt) =========================
# Lee parse_fallbacks.log y resume:
#   - n total predicciones por subtask (de _total_count del parser)
#   - n y % via fallback (parser-no-match O exception)
#   - desglose: % parser-no-match  vs  % por exception (con tipo)
print()
print(f"=== FALLBACK REPORT  | log = {FALLBACK_LOG} ===")

_log_recs = []
if FALLBACK_LOG.exists():
    with open(FALLBACK_LOG, "r", encoding="utf-8") as f:
        for line in f:
            try:
                _log_recs.append(json.loads(line))
            except Exception:
                pass

print(f"  registros en log : {len(_log_recs):,}")

rows = []
for task in ["t31", "t32", "t33"]:
    total_calls   = _total_count[task]
    total_fbacks  = _fallback_count[task]
    task_log      = [r for r in _log_recs if r.get("task") == task]
    n_exc         = sum(1 for r in task_log if r.get("exc_type"))
    n_parser      = max(0, len(task_log) - n_exc)
    rows.append({
        "task"          : task,
        "n_predictions" : total_calls,
        "n_fallback"    : total_fbacks,
        "%_fallback"    : (100*total_fbacks/total_calls) if total_calls else 0.0,
        "n_exception"   : n_exc,
        "%_exception"   : (100*n_exc/total_calls) if total_calls else 0.0,
        "n_parser_miss" : n_parser,
        "%_parser_miss" : (100*n_parser/total_calls) if total_calls else 0.0,
    })

fallback_df = pd.DataFrame(rows)
print()
print(fallback_df.to_string(index=False, float_format="%.2f"))

# Top tipos de excepcion
if _log_recs:
    exc_counter = Counter(r.get("exc_type") for r in _log_recs if r.get("exc_type"))
    if exc_counter:
        print()
        print("--- Top tipos de excepcion ---")
        for k, v in exc_counter.most_common(10):
            print(f"    {k:<28}  {v:>5}")

fallback_df.to_csv(METR_DIR / "fallback_report.csv", index=False)
print(f"\nGuardado en {METR_DIR / 'fallback_report.csv'}")


In [ ]:
# === 8.5 - Variabilidad multi-seed del pool (opcional) =====================
# Solo se ejecuta cuando N_POOL_SEEDS > 1. Para cada seed s en 1..N-1:
#   1) Reconstruye los pools T3.1/T3.2/T3.3 con tiebreaker random=seed.
#   2) Corre la inferencia jerarquica en checkpoints scoped a la seed.
#   3) Evalua con PyEvALL y acumula metricas.
# Al final imprime media +- std de ICM/ICMNorm/FMeasure incluyendo la run
# principal como seed=0.
if N_POOL_SEEDS <= 1 or RUN_MODE != "train_eval":
    print(f"[multi-seed] desactivado: N_POOL_SEEDS={N_POOL_SEEDS}, "
          f"RUN_MODE={RUN_MODE}. Saltando.")
else:
    print(f"=== Multi-seed pool variability: seeds 0..{N_POOL_SEEDS-1} ===")
    seed_rows = []
    # Seed 0 = run principal (ya en results)
    if "results" in dir():
        for task in ["t31","t32","t33"]:
            r = results.get(task, {})
            seed_rows.append({"seed": 0, "task": task,
                              "ICM": r.get("ICM"), "ICMNorm": r.get("ICMNorm"),
                              "FMeasure": r.get("FMeasure")})

    for s in range(1, int(N_POOL_SEEDS)):
        print(f"\n--- seed = {s} ---")
        seed_pools = {
            "t31": select_few_shot_pool_t31(df_train, raw_train, seed=s),
            "t32": select_few_shot_pool_t32(df_train, raw_train, seed=s),
            "t33": select_few_shot_pool_t33(df_train, raw_train, seed=s),
        }
        seed_pool_ids = set()
        for task in ["t31","t32","t33"]:
            for ex in seed_pools[task]: seed_pool_ids.add(str(ex["id"]))

        # Scoping de checkpoints + predicciones por seed
        seed_dir = CKPT_DIR / f"seed{s}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        seed_pred_paths = {t: PRED_DIR / f"pred_{t}_hard_seed{s}_{_CKPT_SUFFIX}.json"
                           for t in ["t31","t32","t33"]}
        # backup global pools/paths
        _orig_pools, _orig_paths = POOLS, CKPT_PATHS
        try:
            globals()["POOLS"] = seed_pools
            globals()["CKPT_PATHS"] = {t: seed_dir / f"predictions_{t}_partial.json"
                                        for t in ["t31","t32","t33"]}

            # Re-corremos la jerarquia con los pools de esta seed
            _eval_t31 = get_eval_df("t31"); _eval_t32 = get_eval_df("t32"); _eval_t33 = get_eval_df("t33")
            _union_ids = (set(_eval_t31["id_EXIST"]) | set(_eval_t32["id_EXIST"])
                          | set(_eval_t33["id_EXIST"]))
            _preds_t31 = run_inference("t31", target_ids=_union_ids)
            _sexist_ids = {vid for vid, v in _preds_t31.items() if v == "YES"}
            _preds_t32 = run_inference("t32", target_ids=_sexist_ids & set(_eval_t32["id_EXIST"]))
            for vid in _eval_t32["id_EXIST"]:
                if _preds_t31.get(vid) == "NO": _preds_t32[vid] = "NO"
            _preds_t33 = run_inference("t33", target_ids=_sexist_ids & set(_eval_t33["id_EXIST"]))
            for vid in _eval_t33["id_EXIST"]:
                if _preds_t31.get(vid) == "NO": _preds_t33[vid] = ["NO"]

            # Volcado de JSONs por seed y evaluacion
            for task, preds in [("t31",_preds_t31),("t32",_preds_t32),("t33",_preds_t33)]:
                pj = build_pyevall_hard(preds, task)
                _sanity_check_pred(pj, task)
                with open(seed_pred_paths[task], "w", encoding="utf-8") as f:
                    json.dump(pj, f, ensure_ascii=False, indent=2)
            for task in ["t31","t32","t33"]:
                pl = json.load(open(seed_pred_paths[task],"r",encoding="utf-8"))
                gl = build_pyevall_gold_hard([p["id"] for p in pl], task)
                m  = evaluate_pyevall_hard(pl, gl, task)
                seed_rows.append({"seed": s, "task": task,
                                  "ICM": m.get("ICM"), "ICMNorm": m.get("ICMNorm"),
                                  "FMeasure": m.get("FMeasure")})
                print(f"  [{task}] seed={s}  ICM={m.get('ICM')}  "
                      f"ICMNorm={m.get('ICMNorm')}  F={m.get('FMeasure')}")
        finally:
            globals()["POOLS"]      = _orig_pools
            globals()["CKPT_PATHS"] = _orig_paths

    if seed_rows:
        seed_df = pd.DataFrame(seed_rows)
        print()
        print(f"=== Multi-seed summary (n_seeds={int(N_POOL_SEEDS)}) ===")
        for task in ["t31","t32","t33"]:
            sub = seed_df[seed_df["task"] == task]
            print(f"[{task.upper()}] (across {len(sub)} seeds)")
            for m in ["ICM","ICMNorm","FMeasure"]:
                vals = sub[m].dropna().astype(float)
                if len(vals) >= 2:
                    print(f"    {m:>10}: mean={vals.mean():.4f}  std={vals.std():.4f}  "
                          f"min={vals.min():.4f}  max={vals.max():.4f}")
                elif len(vals) == 1:
                    print(f"    {m:>10}: {vals.iloc[0]:.4f}  (only 1 seed)")
                else:
                    print(f"    {m:>10}: <missing>")
        seed_df.to_csv(METR_DIR / "metrics_multiseed.csv", index=False)
        print(f"\nGuardado en {METR_DIR / 'metrics_multiseed.csv'}")


## 9 - Inferencia sobre el TEST oficial (on-demand)

Ejecuta inferencia jerarquica sobre los **674 videos de test oficial** y empaqueta la submission, sin necesidad de cambiar `STAGE` ni reejecutar el notebook.

In [28]:
# === 9.1 - Inferencia JERARQUICA sobre TEST + submission ===================
TEAM_NAME = "J3"   # CAMBIA esto por tu nombre de equipo
RUN_ID    = 1                   # 1, 2 o 3

if TEST_JSON is None or not TEST_JSON.exists():
    raise RuntimeError(f"TEST_JSON no localizado: {TEST_JSON}.")
if TEST_VID_DIR is None or not TEST_VID_DIR.exists():
    raise RuntimeError(f"TEST_VID_DIR no localizado: {TEST_VID_DIR}.")

if not (df["split"] == "test").any():
    print("Cargando TEST set en memoria...")
    with open(TEST_JSON, "r", encoding="utf-8", errors="replace") as f:
        raw_test_now = json.load(f)
    rows_test_now = _build_rows(raw_test_now, TEST_VID_DIR, "test")
    df = pd.concat([df, pd.DataFrame(rows_test_now)], ignore_index=True)
    raw.update(raw_test_now)

df_test_now = df[df["split"] == "test"].reset_index(drop=True)
assert df_test_now["vid_ok"].all(), "Hay videos test sin .mp4"
print(f"Test set listo: {len(df_test_now):,} videos")
test_ids = set(df_test_now["id_EXIST"].astype(str))

TEST_CKPT_PATHS = {t: CKPT_DIR / f"predictions_{t}_test_partial.json"
                    for t in ["t31","t32","t33"]}

def _load_test_ckpt(task):
    p = TEST_CKPT_PATHS[task]
    return json.load(open(p, "r", encoding="utf-8")) if p.exists() else {}

def _save_test_ckpt(task, preds):
    with open(TEST_CKPT_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False)

def _run_test_inference(task, target_ids):
    eval_df = df_test_now[df_test_now["id_EXIST"].isin(target_ids)].copy().reset_index(drop=True)
    preds = _load_test_ckpt(task)
    todo = eval_df[~eval_df["id_EXIST"].isin(preds.keys())].copy()
    print()
    print(f"=== TEST {task.upper()} ===")
    print(f"  a predecir   : {len(eval_df):,}")
    print(f"  ya predichos : {len(preds):,}")
    print(f"  pendientes   : {len(todo):,}")
    if len(todo) == 0: return preds
    pbar = tqdm(todo.itertuples(index=False), total=len(todo), desc=f"test {task}")
    n_done = 0
    for row in pbar:
        vid = row.id_EXIST
        try:
            pred = predict_video(vid, row.video_path, row.text_ocr, task, POOLS[task])
        except torch.cuda.OutOfMemoryError as e:
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
            print(); print(f"[OOM] {vid}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<OOM: {e}>", str(pred), exc_type="OutOfMemoryError")
        except FileNotFoundError as e:
            print(); print(f"[FileNotFound] {vid}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<FileNotFoundError: {e}>", str(pred),
                          exc_type="FileNotFoundError")
        except RuntimeError as e:
            print(); print(f"[RuntimeError] {vid}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<RuntimeError: {e}>", str(pred),
                          exc_type="RuntimeError")
        except Exception as e:
            print(); print(f"[err] {vid}: {type(e).__name__}: {e}")
            pred = _task_fallback(task)
            _log_fallback(task, vid, f"<{type(e).__name__}: {e}>", str(pred),
                          exc_type=type(e).__name__)
        preds[vid] = pred
        n_done += 1
        if n_done % CKPT_EVERY == 0:
            _save_test_ckpt(task, preds)
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
    _save_test_ckpt(task, preds)
    return preds

# === FLUJO JERARQUICO TEST ===================================================
print()
print(f"--- Paso 1/3: T3.1 sobre {len(test_ids):,} videos ---")
test_preds_t31 = _run_test_inference("t31", test_ids)
test_sexist_ids = {vid for vid, v in test_preds_t31.items() if v == "YES"}
print()
print(f"=== T3.1 -> {len(test_sexist_ids):,} YES ({100*len(test_sexist_ids)/len(test_ids):.1f}%) ===")

print()
print(f"--- Paso 2/3: T3.2 sobre {len(test_sexist_ids):,} sexistas ---")
test_preds_t32 = _run_test_inference("t32", test_sexist_ids)
for vid in test_ids:
    if test_preds_t31.get(vid) == "NO":
        test_preds_t32[vid] = "NO"
_save_test_ckpt("t32", test_preds_t32)

print()
print(f"--- Paso 3/3: T3.3 sobre {len(test_sexist_ids):,} sexistas ---")
test_preds_t33 = _run_test_inference("t33", test_sexist_ids)
for vid in test_ids:
    if test_preds_t31.get(vid) == "NO":
        test_preds_t33[vid] = ["NO"]
_save_test_ckpt("t33", test_preds_t33)

# JSONs PyEvALL
TEST_PRED_PATHS = {
    "t31": PRED_DIR / "pred_t31_hard_test.json",
    "t32": PRED_DIR / "pred_t32_hard_test.json",
    "t33": PRED_DIR / "pred_t33_hard_test.json",
}
print()
for task, preds in [("t31", test_preds_t31), ("t32", test_preds_t32), ("t33", test_preds_t33)]:
    pj = build_pyevall_hard(preds, task)
    _sanity_check_pred(pj, task)
    with open(TEST_PRED_PATHS[task], "w", encoding="utf-8") as f:
        json.dump(pj, f, ensure_ascii=False, indent=2)
    print(f"[{task}] {len(pj):,} predicciones -> {TEST_PRED_PATHS[task].name}")

for task, preds in [("t31", test_preds_t31), ("t32", test_preds_t32), ("t33", test_preds_t33)]:
    miss = test_ids - set(preds.keys())
    extra = set(preds.keys()) - test_ids
    assert not miss,  f"[{task}] faltan {len(miss)} predicciones"
    assert not extra, f"[{task}] hay {len(extra)} predicciones fuera del test set"
print()
print(f"OK Cobertura test: {len(test_ids):,} videos en cada subtask.")

print()
print("--- Distribucion de predicciones por subtask ---")
for task, preds in [("t31", test_preds_t31), ("t32", test_preds_t32), ("t33", test_preds_t33)]:
    if task == "t33":
        cnt = Counter()
        for v in preds.values():
            key = "NO" if v == ["NO"] else "+".join(sorted(v))
            cnt[key] += 1
    else:
        cnt = Counter(preds.values())
    print()
    print(f"[{task.upper()}]")
    total = sum(cnt.values())
    for k, v in cnt.most_common():
        print(f"    {str(k):<60}  {v:>5}  ({100*v/total:5.1f}%)")

# Empaquetar submission
# Guidelines V0.5 pag. 15 fijan el formato exacto:
#   <task>_<subtask>_<evaluation_context>_<team_name>_<run_id>
# (sin extension). NO ANADIR .json — los ejemplos oficiales son
#   exist2026_UNED/task2_2_hard_UNED_1
#   exist2026_UNED/task3_3_soft_UNED_3
import shutil
SUBM_BASE = SUBM_DIR / f"exist2026_{TEAM_NAME}"
if SUBM_BASE.exists(): shutil.rmtree(SUBM_BASE)
SUBM_BASE.mkdir(parents=True, exist_ok=True)
name_map = {"t31": "task3_1", "t32": "task3_2", "t33": "task3_3"}
print()
print(f"--- Empaquetando submission en {SUBM_BASE.name}/ ---")
for task in ["t31","t32","t33"]:
    src = TEST_PRED_PATHS[task]
    dst = SUBM_BASE / f"{name_map[task]}_hard_{TEAM_NAME}_{RUN_ID}"
    dst.write_bytes(src.read_bytes())
    print(f"  {dst.name}  ({src.stat().st_size/1024:.1f} KB)")

zip_path = SUBM_DIR / f"exist2026_{TEAM_NAME}.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip",
                     root_dir=str(SUBM_DIR),
                     base_dir=f"exist2026_{TEAM_NAME}")
print()
print(f"OK Submission empaquetada: {zip_path}")
print(f"   Sube este ZIP al form: https://forms.gle/5hY91c7aBv563oZM7")
if TEAM_NAME == "TODO_TEAM_NAME":
    print()
    print("ATENCION: cambia TEAM_NAME por tu nombre real de equipo y reejecuta esta celda.")

Cargando TEST set en memoria...
Test set listo: 674 videos

--- Paso 1/3: T3.1 sobre 674 videos ---

=== TEST T31 ===
  a predecir   : 674
  ya predichos : 0
  pendientes   : 674


test t31:   0%|          | 0/674 [00:00<?, ?it/s]

2026-05-04 15:47:41,769 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 15:47:41,820 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - torchcodec:  video_path='/content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/videos/7164058026352168197.mp4', total_frames=394, video_fps=25.0, time=0.051s
2026-05-04 15:47:41,824 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 15:47:41,858 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - torchcodec:  video_path='/content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/videos/7248606026386263323.mp4', total_frames=204, video_fps=30.0, time=0.034s
2026-05-04 15:47:41,863 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 15:47:42,541 - qwen_vl_utils.vision_process - INFO

test t32:   0%|          | 0/195 [00:00<?, ?it/s]

2026-05-04 15:59:52,811 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 15:59:52,862 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - torchcodec:  video_path='/content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/videos/7164058026352168197.mp4', total_frames=394, video_fps=25.0, time=0.051s
2026-05-04 15:59:52,866 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 15:59:53,174 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - torchcodec:  video_path='/content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/videos/7198351362462108934.mp4', total_frames=1422, video_fps=30.0, time=0.308s
2026-05-04 15:59:53,193 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 15:59:53,262 - qwen_vl_utils.vision_process - INF

test t33:   0%|          | 0/195 [00:00<?, ?it/s]

2026-05-04 16:02:34,885 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 16:02:35,108 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - torchcodec:  video_path='/content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/videos/7028391588913106182.mp4', total_frames=989, video_fps=29.116781, time=0.222s
2026-05-04 16:02:35,123 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 16:02:35,172 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - torchcodec:  video_path='/content/drive/MyDrive/EXIST_2026/EXIST 2026 Dataset V0.1/EXIST 2026 Videos Dataset/training/videos/7318400739775204614.mp4', total_frames=401, video_fps=30.0, time=0.048s
2026-05-04 16:02:35,178 - qwen_vl_utils.vision_process - INFO - _read_video_torchcodec() - set TORCHCODEC_NUM_THREADS: 8
2026-05-04 16:02:35,277 - qwen_vl_utils.vision_process -